# A Comparative Evaluation of Machine Learning and Deep Learning Models for Darknet Traffic Classification

## Stage 1 - Dataset Loading, Exploratory Data Analysis, Cleaning, Preprocessing
## Stage 2 - Model Development, Evaluation and Proposal and Feature Engineering

**Dataset:** CIC-Darknet2020 &nbsp;|&nbsp; **Module:** CMP7200 - Individual Master's Project

---

 The project frames darknet detection as a **binary classification problem** - is a given network flow darknet (anonymised via Tor or VPN) or ordinary traffic? - because that coarse decision is the operational gatekeeping step and the point at which class imbalance bites hardest.

CIC-Darknet2020 amalgamates the ISCXTor2016 and ISCXVPN2016 collections, describing each flow with roughly 85 statistical features (Habibi Lashkari, Kaur and Rahali, 2020). The corpus is known to carry duplicated records, missing and infinite values, label inconsistencies and severe class imbalance. My proposal treats the disciplined handling of these defects as part of the contribution rather than a preliminary chore, and this notebook documents every decision I take.

**Contents**

1. **Dataset Loading** - acquisition via `kagglehub`, structural overview
2. **Exploratory Data Analysis** - labels and imbalance, data quality, distributions, correlations, outliers
3. **Data Cleaning and Preprocessing** - identifier removal, label correction, missing/infinite handling, deduplication, encoding
4. **Feature Engineering** - derived features, transformation, redundancy pruning, model-free relevance ranking, split and scaling
5. **Proposed Machine Learning Algorithms** - the modelling plan for the next stage (no models are built or trained here)

## 0. Environment Setup

I fix the random seed globally so that every sampled or split quantity in this notebook is reproducible, and I define a single colour scheme up front so that each class keeps the same colour in every figure (colour follows the entity, never the chart).

In [1]:
import json
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

# ---- fixed colour assignments (consistent across every figure) ----
INK, INK2, MUTED, GRID = "#0b0b0b", "#52514e", "#898781", "#e1e0d9"

TRAFFIC_COLOURS = {          # 4-way traffic type: cool = ordinary, warm = anonymised
    "Non-Tor": "#2a78d6",
    "NonVPN":  "#1baf7a",
    "VPN":     "#eda100",
    "Tor":     "#e34948",
}
BINARY_COLOURS = {"Benign": "#2a78d6", "Darknet": "#e34948"}

APP_SLOTS = ["#2a78d6", "#1baf7a", "#eda100", "#008300",
             "#4a3aa7", "#e34948", "#e87ba4", "#eb6834"]

# diverging map for correlations: red (negative) -> neutral -> blue (positive)
DIVERGING = LinearSegmentedColormap.from_list("div", ["#e34948", "#f0efec", "#2a78d6"])

plt.rcParams.update({
    "figure.dpi": 100, "figure.facecolor": "white",
    "axes.facecolor": "white", "axes.edgecolor": "#c3c2b7",
    "axes.labelcolor": INK2, "axes.titlecolor": INK,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold",
})

print("numpy ", np.__version__)
print("pandas", pd.__version__)

numpy  1.26.3
pandas 2.2.0


# 1. Dataset Loading

I obtain CIC-Darknet2020 through `kagglehub`, which downloads the public Kaggle mirror of the CIC distribution and caches it locally. The archive contains a single file, `Darknet.CSV` (~73 MB), with one row per bidirectional network flow.

One practical defect of this distribution is worth stating before loading: a small number of physical lines in the CSV are corrupted (they contain more fields than the 85-column header declares, apparently the result of two records colliding on one line). These lines cannot be attributed to any flow reliably, so I skip them at parse time and report exactly how many were lost.

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("peterfriedrich1/cicdarknet2020-internet-traffic")
print("Path to dataset files:", path)

csv_path = next(Path(path).glob("*.CSV"))
print("CSV file:", csv_path.name, f"({csv_path.stat().st_size / 1e6:.1f} MB)")

Path to dataset files: /home/claude-user/.cache/kagglehub/datasets/peterfriedrich1/cicdarknet2020-internet-traffic/versions/1
CSV file: Darknet.CSV (73.3 MB)


In [3]:
# count physical data lines so the number of skipped (malformed) lines can be reported
with open(csv_path, encoding="utf-8", errors="replace") as f:
    physical_lines = sum(1 for _ in f) - 1   # minus header

df_raw = pd.read_csv(csv_path, low_memory=False, on_bad_lines="skip")

print(f"Physical data lines in file : {physical_lines:,}")
print(f"Rows parsed successfully    : {len(df_raw):,}")
print(f"Malformed lines skipped     : {physical_lines - len(df_raw):,}")

Physical data lines in file : 141,531
Rows parsed successfully    : 141,530
Malformed lines skipped     : 1


Only a single physically corrupted line is lost - a negligible cost, and a row that could not have been trusted in any case. Everything else parses into the expected 85-column layout.

In [4]:
print("Dataset shape:", df_raw.shape)
df_raw.head()

Dataset shape: (141530, 85)


,Flow ID,Src IP,Src Port,Dst IP,Dst Port,Protocol,Timestamp,Flow Duration,Total Fwd Packet,Total Bwd packets,...,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,Label.1
0,10.152.152.11-216.58.220.99-57158-443-6,10.152.152.11,57158,216.58.220.99,443,6,24/07/2015 04:09:48 PM,229,1,1,...,0,0,0,0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,Non-Tor,AUDIO-STREAMING
1,10.152.152.11-216.58.220.99-57159-443-6,10.152.152.11,57159,216.58.220.99,443,6,24/07/2015 04:09:48 PM,407,1,1,...,0,0,0,0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,Non-Tor,AUDIO-STREAMING
2,10.152.152.11-216.58.220.99-57160-443-6,10.152.152.11,57160,216.58.220.99,443,6,24/07/2015 04:09:48 PM,431,1,1,...,0,0,0,0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,Non-Tor,AUDIO-STREAMING
3,10.152.152.11-74.125.136.120-49134-443-6,10.152.152.11,49134,74.125.136.120,443,6,24/07/2015 04:09:48 PM,359,1,1,...,0,0,0,0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,Non-Tor,AUDIO-STREAMING
4,10.152.152.11-173.194.65.127-34697-19305-6,10.152.152.11,34697,173.194.65.127,19305,6,24/07/2015 04:09:45 PM,10778451,591,400,...,0,0,0,0,1.437765e+15,3.117718e+06,1.437765e+15,1.437765e+15,Non-Tor,AUDIO-STREAMING


In [5]:
# data types and column inventory
dtype_summary = df_raw.dtypes.astype(str).value_counts().rename_axis("dtype").to_frame("columns")
display(dtype_summary)

print("Column inventory (85 columns):\n")
for i, (col, dt) in enumerate(zip(df_raw.columns, df_raw.dtypes.astype(str)), 1):
    print(f"{i:>3}. {col:<32} {dt}")

,columns
dtype,
int64,55
float64,24
object,6


Column inventory (85 columns):

  1. Flow ID                          object
  2. Src IP                           object
  3. Src Port                         int64
  4. Dst IP                           object
  5. Dst Port                         int64
  6. Protocol                         int64
  7. Timestamp                        object
  8. Flow Duration                    int64
  9. Total Fwd Packet                 int64
 10. Total Bwd packets                int64
 11. Total Length of Fwd Packet       int64
 12. Total Length of Bwd Packet       int64
 13. Fwd Packet Length Max            int64
 14. Fwd Packet Length Min            int64
 15. Fwd Packet Length Mean           float64
 16. Fwd Packet Length Std            float64
 17. Bwd Packet Length Max            int64
 18. Bwd Packet Length Min            int64
 19. Bwd Packet Length Mean           float64
 20. Bwd Packet Length Std            float64
 21. Flow Bytes/s                     float64
 22. Flow Packets/s           

**Structure of the data.** The 85 columns fall into three groups:

- **Flow identifiers (7):** `Flow ID`, `Src IP`, `Src Port`, `Dst IP`, `Dst Port`, `Protocol`, `Timestamp`. These identify *which* conversation the flow belongs to rather than describing its statistical shape.
- **Flow-statistic features (76):** packet counts, byte counts, packet-length statistics, inter-arrival times (IAT), TCP flag counts, bulk-transfer rates, window sizes, and active/idle timings - the observable "shape" of encrypted traffic that the proposal identifies as the only exploitable signal.
- **Labels (2):** `Label` - the four-way traffic type (**Non-Tor, NonVPN, VPN, Tor**), from which I derive the binary darknet target; and `Label.1` - the application category (browsing, streaming, chat, …), which I retain for secondary analysis.

Most columns are numeric (`int64`/`float64`); the six text columns are the identifiers and the two labels. The `Timestamp` column is stored as text and, notably, spans captures from 2015-2016 - a reminder that this corpus was stitched together from two older collections, which is precisely why merge artefacts such as duplicates need attention.

# 2. Exploratory Data Analysis

The EDA is organised around the questions that matter for the modelling stage: *how imbalanced are the classes?* *how dirty is the data?* *which features separate darknet from ordinary traffic?* and *how are the features distributed and inter-related?* Each subsection states its finding explicitly, and every finding feeds a concrete decision in Sections 3-4.

## 2.1 Target Labels and Class Imbalance

The four-way `Label` is the primary label. My binary target treats **Tor + VPN as darknet (positive class)** and **Non-Tor + NonVPN as benign** - the framing used by DarkDetect (Sarwar et al., 2021) and adopted in my proposal.

In [6]:
label_counts = df_raw["Label"].value_counts()
display(label_counts.to_frame("flows").assign(share=lambda d: (d["flows"] / len(df_raw) * 100).round(2)))

darknet_mask = df_raw["Label"].isin(["Tor", "VPN"])
binary_counts = pd.Series({"Benign": int((~darknet_mask).sum()), "Darknet": int(darknet_mask.sum())})

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))

order = label_counts.index.tolist()
bars = axes[0].bar(order, label_counts[order],
                   color=[TRAFFIC_COLOURS[c] for c in order], width=0.62)
axes[0].bar_label(bars, fmt="{:,.0f}", padding=3, fontsize=9, color=INK2)
axes[0].set_title("Traffic type (4-way label)")
axes[0].set_ylabel("Flows")
axes[0].grid(axis="x", visible=False)

bars = axes[1].bar(binary_counts.index, binary_counts.values,
                   color=[BINARY_COLOURS[c] for c in binary_counts.index], width=0.5)
axes[1].bar_label(bars, labels=[f"{v:,} ({v/len(df_raw):.1%})" for v in binary_counts.values],
                  padding=3, fontsize=9, color=INK2)
axes[1].set_title("Binary detection target")
axes[1].grid(axis="x", visible=False)

fig.tight_layout()
plt.show()

print(f"Binary imbalance ratio (benign : darknet) = {binary_counts['Benign'] / binary_counts['Darknet']:.2f} : 1")
print(f"Rarest class (Tor) share of all flows     = {label_counts['Tor'] / len(df_raw):.2%}")

,flows,share
Label,,
Non-Tor,93356,65.96
NonVPN,23863,16.86
VPN,22919,16.19
Tor,1392,0.98


Binary imbalance ratio (benign : darknet) = 4.82 : 1
Rarest class (Tor) share of all flows     = 0.98%


**Findings - imbalance is severe and two-layered.**

- At the binary level, darknet flows make up only **~17%** of the corpus (about **4.8 : 1** benign-to-darknet). A classifier that always predicts "benign" would already score ~83% accuracy, which is why my proposal insists on macro-averaged F1, recall and ROC-AUC rather than accuracy alone.
- Inside the darknet class the imbalance is worse: **Tor is under 1% of all flows** (~1,400 records vs ~93,000 Non-Tor). Tor is exactly the traffic an analyst most needs to catch, so minority-class recall is the explicit design target of my proposed model.
- These observations directly justify the class-balancing strategy (SMOTE plus class-weighted loss, applied only inside training folds) planned in Section 7.2 of the proposal and documented in Section 3.8 below.

In [7]:
# application-level label (Label.1)
app_counts = df_raw["Label.1"].value_counts()

# canonical (case-normalised) app -> colour slot, so case-variants share a hue
canon = lambda s: s.strip().title()
canon_apps = sorted({canon(a) for a in app_counts.index})
app_colour = {a: APP_SLOTS[i % len(APP_SLOTS)] for i, a in enumerate(canon_apps)}

fig, ax = plt.subplots(figsize=(11, 4))
bars = ax.bar(app_counts.index, app_counts.values,
              color=[app_colour[canon(a)] for a in app_counts.index], width=0.62)
ax.bar_label(bars, fmt="{:,.0f}", padding=3, fontsize=8, color=INK2)
ax.set_title("Application label (Label.1) - raw values as stored in the file")
ax.set_ylabel("Flows")
ax.grid(axis="x", visible=False)
plt.xticks(rotation=30, ha="right")
fig.tight_layout()
plt.show()

print("Raw distinct values :", app_counts.index.tolist())
print("Case-normalised set :", canon_apps)

Raw distinct values : ['P2P', 'Browsing', 'Audio-Streaming', 'Chat', 'File-Transfer', 'Video-Streaming', 'Email', 'VOIP', 'AUDIO-STREAMING', 'Video-streaming', 'File-transfer']
Case-normalised set : ['Audio-Streaming', 'Browsing', 'Chat', 'Email', 'File-Transfer', 'P2P', 'Video-Streaming', 'Voip']


**Finding - the application label is internally inconsistent.** The file stores **11 distinct strings** for what are really **8 application categories**: `AUDIO-STREAMING` vs `Audio-Streaming`, `Video-streaming` vs `Video-Streaming`, and `File-transfer` vs `File-Transfer` are case-variants of the same class (the bars above share a colour when they are the same underlying category). Left uncorrected, this would silently split classes during any grouped analysis or secondary categorisation task. I repair this in Section 3.2.

## 2.2 Missing and Infinite Values

Flow exporters divide by the flow duration to produce rate features, so zero-duration flows generate `NaN` and `±inf` values. Both are documented weaknesses of this corpus and both break scaling and most learning algorithms, so I quantify them exactly.

In [8]:
num_cols_raw = df_raw.select_dtypes(include=[np.number]).columns

missing = df_raw.isna().sum()
infinite = np.isinf(df_raw[num_cols_raw]).sum().reindex(df_raw.columns, fill_value=0)

quality = pd.DataFrame({"missing (NaN)": missing, "infinite (±inf)": infinite})
quality = quality[(quality > 0).any(axis=1)]
display(quality)

affected = df_raw[num_cols_raw].isna().any(axis=1) | np.isinf(df_raw[num_cols_raw]).any(axis=1)
print(f"Rows affected by NaN or ±inf : {affected.sum():,}  ({affected.sum() / len(df_raw):.3%} of the data)")
print(f"Flow Duration of affected rows - max: {df_raw.loc[affected, 'Flow Duration'].max()}")

,missing (NaN),infinite (±inf)
Flow Bytes/s,47,2
Flow Packets/s,0,49


Rows affected by NaN or ±inf : 49  (0.035% of the data)
Flow Duration of affected rows - max: 0


**Findings.** Only two columns are affected - `Flow Bytes/s` (47 `NaN` + 2 `inf`) and `Flow Packets/s` (49 `inf`) - and the defects co-occur on the **same ~50 rows**, all of which are zero-duration flows (the maximum `Flow Duration` among affected rows is 0). These are degenerate captures rather than meaningful traffic, they amount to well under 0.1% of the corpus, and imputing a rate for a zero-length flow would be fabricating a value with no physical meaning. The principled treatment, per Section 7.2 of my proposal, is **documented row removal**, applied in Section 3.3.

## 2.3 Duplicate Records

Duplication is the most consequential defect in CIC-Darknet2020. As Saleem, Islam and Islam (2024) note, naive random splitting can leak near-identical records across train and test sets, quietly inflating reported results - one of the main reasons headline accuracies in this literature are hard to trust. I measure duplication two ways: exact full-row duplicates, and duplicates after ignoring the identifier columns (two flows with identical statistics but different IPs/timestamps are still, from the model's point of view, the same training example).

In [9]:
exact_dups = df_raw.duplicated().sum()

id_cols = ["Flow ID", "Src IP", "Src Port", "Dst IP", "Timestamp"]
feature_view = df_raw.drop(columns=id_cols)
feature_dups = feature_view.duplicated().sum()

print(f"Exact full-row duplicates                      : {exact_dups:,}  ({exact_dups / len(df_raw):.2%})")
print(f"Duplicates ignoring identifier columns         : {feature_dups:,}  ({feature_dups / len(df_raw):.2%})")

# duplicate burden per traffic type (feature-level)
dup_by_label = (feature_view[feature_view.duplicated(keep=False)]["Label"]
                .value_counts()
                .reindex(TRAFFIC_COLOURS.keys()))
fig, ax = plt.subplots(figsize=(7, 3.2))
bars = ax.barh(dup_by_label.index[::-1], dup_by_label.values[::-1],
               color=[TRAFFIC_COLOURS[c] for c in dup_by_label.index[::-1]], height=0.6)
ax.bar_label(bars, fmt="{:,.0f}", padding=3, fontsize=9, color=INK2)
ax.set_title("Rows involved in feature-level duplicate groups, by traffic type")
ax.set_xlabel("Flows")
ax.grid(axis="y", visible=False)
fig.tight_layout()
plt.show()

Exact full-row duplicates                      : 24,457  (17.28%)
Duplicates ignoring identifier columns         : 34,198  (24.16%)


**Findings.** Roughly **17%** of rows are exact duplicates, rising to about **24%** once identifiers are ignored - nearly a quarter of the corpus is redundant. Duplication is not uniform across classes, so it also distorts the apparent class balance. Any model evaluated on a random split of the uncleaned data would be partially tested on its own training rows. Deduplication (Section 3.4) is therefore a validity requirement, not housekeeping - exactly why my proposal treats preprocessing as a first-class contribution.

## 2.4 Statistical Summaries and Skewness

Flow statistics span enormous ranges (microsecond timings to multi-megabyte byte counts), so I inspect summary statistics and skewness before choosing transformations.

In [10]:
key_feats = ["Flow Duration", "Total Fwd Packet", "Total Bwd packets",
             "Flow Bytes/s", "Flow Packets/s", "Packet Length Mean",
             "Flow IAT Mean", "Idle Mean"]
display(df_raw[key_feats].describe().T.round(2))

skew = df_raw[num_cols_raw].skew().sort_values(ascending=False)
print("Ten most right-skewed features:")
display(skew.head(10).round(1).to_frame("skewness"))

,count,mean,std,min,25%,50%,75%,max
Flow Duration,141530.0,2.081280e+07,3.809155e+07,0.00,17781.00,4.162820e+05,1.181470e+07,1.200000e+08
Total Fwd Packet,141530.0,1.528000e+02,2.378320e+03,1.00,1.00,2.000000e+00,4.000000e+00,2.381610e+05
Total Bwd packets,141530.0,1.546400e+02,3.418720e+03,0.00,0.00,1.000000e+00,3.000000e+00,4.708620e+05
Flow Bytes/s,141483.0,inf,NaN,0.00,0.00,7.408000e+01,9.908300e+02,inf
Flow Packets/s,141530.0,inf,NaN,0.02,0.64,7.300000e+00,4.006800e+02,inf
Packet Length Mean,141530.0,9.437000e+01,1.905600e+02,0.00,0.00,2.836000e+01,9.584000e+01,6.647070e+03
Flow IAT Mean,141530.0,2.604871e+06,7.124917e+06,0.00,3021.50,2.071620e+05,1.882400e+06,1.199856e+08
Idle Mean,141530.0,7.027739e+14,7.058343e+14,0.00,0.00,7.281252e+14,1.456263e+15,1.456417e+15


Ten most right-skewed features:


,skewness
Down/Up Ratio,299.3
Packet Length Variance,132.5
Total Length of Fwd Packet,126.1
Total Length of Bwd Packet,95.3
Total Bwd packets,83.0
Bwd Header Length,79.1
Bwd Packet/Bulk Avg,76.7
Bwd Bulk Rate Avg,73.8
ACK Flag Count,69.9
RST Flag Count,69.2


**Findings.** The features are **extremely heavy-tailed**: skewness values in the tens-to-hundreds (e.g. `Down/Up Ratio` ≈ 300, `Packet Length Variance` ≈ 133) mean the mean is orders of magnitude above the median for many columns. Two consequences follow. First, distance-based or gradient-based learners would be dominated by a few giant flows unless features are transformed and scaled - motivating the log-transform in Section 4.2 and standardisation in Section 4.5. Second, "outliers" here are a property of the domain (elephant flows genuinely exist), which shapes my outlier-treatment decision in Section 3.6.

## 2.5 Feature Distributions by Class

Do darknet flows actually *look* different? I overlay class-conditional distributions (on a `log1p` axis, given the skew) for six operationally interpretable features.

In [11]:
dist_feats = ["Flow Duration", "Flow Bytes/s", "Packet Length Mean",
              "Flow IAT Mean", "Bwd Packet Length Mean", "FWD Init Win Bytes"]

fig, axes = plt.subplots(2, 3, figsize=(13, 6.5))
for ax, feat in zip(axes.ravel(), dist_feats):
    vals = df_raw[feat].replace([np.inf, -np.inf], np.nan)
    for cls, mask in [("Benign", ~darknet_mask), ("Darknet", darknet_mask)]:
        ax.hist(np.log1p(vals[mask].dropna().clip(lower=0)), bins=60, density=True,
                alpha=0.55, color=BINARY_COLOURS[cls], label=cls,
                edgecolor="white", linewidth=0.3)
    ax.set_title(feat)
    ax.set_xlabel("log1p(value)")
axes[0, 0].set_ylabel("Density")
axes[1, 0].set_ylabel("Density")
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper right", frameon=False, ncols=2)
fig.suptitle("Class-conditional feature distributions (log1p scale)", fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

**Findings.** The classes are separable but not trivially so:

- **Darknet flows are strongly bimodal in duration and inter-arrival time** - a cluster of very short flows (circuit setup, keep-alives) plus a cluster of long-lived tunnels - where benign traffic is more evenly spread.
- **Packet-length statistics separate the classes visibly**: anonymity overlays repackage traffic into cell-like units (Tor famously uses fixed-size cells), which narrows the packet-length distribution relative to ordinary application traffic.
- **`FWD Init Win Bytes` shows distinct class-specific modes**, reflecting the characteristic TCP stacks of tunnel endpoints.

No single feature is decisive - the signal is spread across timing, size and protocol-behaviour features jointly. This supports the proposal's premise that multivariate learners (and architectures that model feature interactions) are needed, rather than simple thresholds.

## 2.6 Feature Behaviour Across the Four Traffic Types

The binary view hides structure inside each class, so I also inspect the four-way label for a few discriminative features.

In [12]:
box_feats = ["Flow Duration", "Packet Length Mean", "Flow IAT Mean", "Bwd Init Win Bytes"]
plot_df = df_raw[box_feats + ["Label"]].replace([np.inf, -np.inf], np.nan).dropna()

fig, axes = plt.subplots(1, 4, figsize=(13.5, 4))
order = list(TRAFFIC_COLOURS.keys())
for ax, feat in zip(axes, box_feats):
    sns.boxplot(data=plot_df.assign(v=np.log1p(plot_df[feat].clip(lower=0))),
                x="Label", y="v", hue="Label", order=order, hue_order=order,
                palette=TRAFFIC_COLOURS, legend=False, ax=ax,
                fliersize=1, linewidth=0.9, flierprops={"alpha": 0.25})
    ax.set_title(feat)
    ax.set_xlabel("")
    ax.set_ylabel("log1p(value)" if feat == box_feats[0] else "")
    ax.tick_params(axis="x", rotation=25)
fig.suptitle("Feature distributions by traffic type (log1p scale)", fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

**Findings.** Tor and VPN are **not interchangeable sub-classes**. In these captures, Tor flows are strikingly long-lived - their duration distribution sits almost entirely at the top of the range, consistent with persistent circuits carrying multiplexed streams - and their mean packet length is both the *highest* and the most *tightly compressed* of the four types, the signature of traffic repackaged into uniform cells. Their backward TCP windows sit near saturation. VPN flows, by contrast, spread across far shorter durations with much wider packet-length variability. This heterogeneity inside the positive class - combined with Tor's tiny sample size - is exactly the condition under which aggregate metrics mislead, and it strengthens the case for the minority-class-focused evaluation my proposal commits to.

## 2.7 Protocol Composition

`Protocol` is the IANA protocol number (6 = TCP, 17 = UDP, 0 = an exporter artefact for flows without a parsed transport header).

In [13]:
proto_names = {6: "TCP", 17: "UDP", 0: "Other (0)"}
proto = (pd.crosstab(np.where(darknet_mask, "Darknet", "Benign"),
                     df_raw["Protocol"].map(proto_names), normalize="index") * 100)
proto = proto[["TCP", "UDP", "Other (0)"]]

fig, ax = plt.subplots(figsize=(8, 2.8))
left = np.zeros(len(proto))
seg_colours = {"TCP": "#2a78d6", "UDP": "#eda100", "Other (0)": "#898781"}
for seg in proto.columns:
    bars = ax.barh(proto.index, proto[seg], left=left, color=seg_colours[seg],
                   label=seg, height=0.55, edgecolor="white", linewidth=2)
    for i, (v, l) in enumerate(zip(proto[seg], left)):
        if v > 6:
            ax.text(l + v / 2, i, f"{v:.1f}%", ha="center", va="center",
                    fontsize=9, color="white", fontweight="bold")
    left += proto[seg].values
ax.set_xlim(0, 100)
ax.set_xlabel("Share of class (%)")
ax.set_title("Transport protocol composition per class")
ax.grid(visible=False)
ax.legend(frameon=False, loc="center left", bbox_to_anchor=(1.01, 0.5))
fig.tight_layout()
plt.show()

print(df_raw["Protocol"].map(proto_names).value_counts().to_string())

Protocol
TCP          84321
UDP          56410
Other (0)      799


**Findings.** The two classes have near mirror-image protocol profiles: benign traffic is roughly two-thirds TCP, while darknet traffic is about 71% **UDP**. The darknet class is numerically dominated by the VPN captures (OpenVPN-style UDP tunnels, and UDP-heavy P2P/streaming carried inside them); Tor is TCP-only but contributes barely a sliver of the class, so it cannot pull the aggregate towards TCP. Protocol is therefore genuinely informative, though a model could over-rely on it as a capture-specific shortcut - a caution worth carrying into the error analysis at the modelling stage. A small number of `Protocol = 0` artefact rows exist and survive on their merits as valid flow statistics; `Protocol` is retained as a feature.

## 2.8 Correlation Structure

Flow exporters compute many arithmetically related statistics, so strong collinearity is expected. Collinearity wastes model capacity, destabilises feature importances, and inflates dimensionality without adding information.

In [14]:
corr = df_raw[num_cols_raw].replace([np.inf, -np.inf], np.nan).corr()

fig, ax = plt.subplots(figsize=(13, 10.5))
sns.heatmap(corr, cmap=DIVERGING, vmin=-1, vmax=1, center=0, square=True,
            cbar_kws={"shrink": 0.6, "label": "Pearson r"}, ax=ax,
            xticklabels=True, yticklabels=True)
ax.set_title("Pearson correlation across the numeric features")
ax.tick_params(labelsize=5.5)
plt.show()

# enumerate near-duplicate pairs
pairs = []
cols = corr.columns
cm = corr.values
for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        if abs(cm[i, j]) > 0.95:
            pairs.append((cols[i], cols[j], round(cm[i, j], 3)))
print(f"Feature pairs with |r| > 0.95: {len(pairs)}\n")
for a, b, r in sorted(pairs, key=lambda t: -abs(t[2])):
    print(f"  {a:<28} <-> {b:<28} r = {r}")

Feature pairs with |r| > 0.95: 12

  Fwd Packet Length Mean       <-> Fwd Segment Size Avg         r = 1.0
  Bwd Packet Length Mean       <-> Bwd Segment Size Avg         r = 1.0
  Packet Length Mean           <-> Average Packet Size          r = 0.993
  Total Bwd packets            <-> Bwd Header Length            r = 0.991
  Idle Mean                    <-> Idle Max                     r = 0.988
  Flow Duration                <-> Fwd IAT Total                r = 0.987
  Total Fwd Packet             <-> Fwd Header Length            r = 0.981
  Bwd Packet Length Mean       <-> Subflow Bwd Bytes            r = 0.981
  Bwd Segment Size Avg         <-> Subflow Bwd Bytes            r = 0.981
  Flow IAT Max                 <-> Fwd IAT Max                  r = 0.962
  Total Bwd packets            <-> ACK Flag Count               r = 0.952
  Bwd Header Length            <-> ACK Flag Count               r = 0.952


**Findings.** Several pairs are perfectly or near-perfectly correlated - some by construction (`Fwd Packet Length Mean` ≡ `Fwd Segment Size Avg` at *r* = 1.0, `Packet Length Mean` ≈ `Average Packet Size`), others because one statistic dominates another (`Flow Duration` ≈ `Fwd IAT Total`). Correlated blocks are visible along the diagonal (the IAT family, the packet-length family, the idle-time family). I prune one member of each redundant pair in Section 4.3, after the log-transform, so the decision is made on the feature values the models will actually see.

## 2.9 Outlier Assessment

I quantify outliers with the standard IQR rule (values beyond 1.5 × IQR from the quartiles) - not to delete them yet, but to size the phenomenon before deciding treatment.

In [15]:
Q1 = df_raw[num_cols_raw].quantile(0.25)
Q3 = df_raw[num_cols_raw].quantile(0.75)
IQR = Q3 - Q1
lo, hi = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

outlier_pct = (((df_raw[num_cols_raw] < lo) | (df_raw[num_cols_raw] > hi)).mean() * 100).sort_values(ascending=False)
top = outlier_pct.head(15)

fig, ax = plt.subplots(figsize=(8.5, 5))
bars = ax.barh(top.index[::-1], top.values[::-1], color="#2a78d6", height=0.62)
ax.bar_label(bars, fmt="%.1f%%", padding=3, fontsize=8, color=INK2)
ax.set_title("Features with the highest IQR-outlier share (top 15)")
ax.set_xlabel("Rows flagged as outliers (%)")
ax.grid(axis="y", visible=False)
fig.tight_layout()
plt.show()

any_outlier = ((df_raw[num_cols_raw] < lo) | (df_raw[num_cols_raw] > hi)).any(axis=1)
print(f"Rows containing at least one IQR outlier: {any_outlier.sum():,} ({any_outlier.mean():.1%})")
print(f"Share of DARKNET rows flagged            : {any_outlier[darknet_mask].mean():.1%}")
print(f"Share of BENIGN rows flagged             : {any_outlier[~darknet_mask].mean():.1%}")

Rows containing at least one IQR outlier: 132,266 (93.5%)
Share of DARKNET rows flagged            : 98.3%
Share of BENIGN rows flagged             : 92.4%


**Findings.** Under the IQR rule, individual features flag up to ~25% of rows and the **large majority of all rows contain at least one flagged value** - with darknet rows flagged *more* often than benign ones. This confirms what Section 2.4 suggested: in flow data, extreme values are the population, not contamination. Deleting or aggressively clipping them would (a) discard a large share of the corpus and (b) disproportionately remove the minority class I most need to detect. The treatment decision is documented in Section 3.6.

## 2.10 Summary of EDA Findings

| # | Finding | Consequence |
|---|---------|-------------|
| 1 | Binary imbalance ~4.8 : 1; Tor is < 1% of flows | Macro/minority metrics; SMOTE + class weights (train folds only) |
| 2 | `Label.1` stores 8 categories under 11 case-variant strings | Canonical label repair (§3.2) |
| 3 | ~50 zero-duration rows carry all NaN/±inf values | Documented row removal (§3.3) |
| 4 | ~24% of rows are duplicates at the feature level | Deduplicate before any split to prevent leakage (§3.4) |
| 5 | Several features are constant or exporter artefacts | Zero-variance removal (§3.5) |
| 6 | Extreme skew (up to ~300); heavy tails are genuine traffic | log1p transform (§4.2); retain outliers (§3.6) |
| 7 | 12 feature pairs with \|r\| > 0.95 | Redundancy pruning (§4.3) |
| 8 | Signal is multivariate - timing + size + protocol behaviour jointly | Supports multivariate ML/DL modelling plan (§5) |

# 3. Data Cleaning and Preprocessing

I now execute the cleaning pipeline specified in Section 7.2 of my proposal. The steps run in a deliberate order - identifiers out first (so deduplication operates on what models actually see), then value repairs, then deduplication, then encoding. Throughout, one principle governs the design: **nothing may be fitted on, or influenced by, data that will later serve as test data.** Scaling is therefore *defined* here but *fitted* only on the training partition (Section 4.5), and resampling (SMOTE) is deferred entirely to the modelling stage where it belongs inside cross-validation folds.

## 3.1 Removing Identifier Columns

`Flow ID`, `Src IP`, `Dst IP`, `Timestamp` and `Src Port` identify *specific conversations in a specific 2015-16 capture*, not properties of darknet traffic. Keeping them would invite the model to memorise the capture (specific hosts and times) rather than learn traffic behaviour - a classic shortcut that cannot generalise. `Src Port` is ephemeral (chosen at random by the client OS) and carries no transferable signal. I retain **`Dst Port`** and **`Protocol`**: destination port is a genuine service indicator (e.g. 443, 9001) and protocol is a behavioural property of the tunnel - both legitimately observable in deployment.

In [16]:
df = df_raw.copy()

id_cols = ["Flow ID", "Src IP", "Dst IP", "Timestamp", "Src Port"]
df = df.drop(columns=id_cols)
print(f"Dropped {len(id_cols)} identifier columns: {id_cols}")
print(f"Shape: {df.shape}")

Dropped 5 identifier columns: ['Flow ID', 'Src IP', 'Dst IP', 'Timestamp', 'Src Port']
Shape: (141530, 80)


## 3.2 Correcting Inconsistent Application Labels

Section 2.1 showed that `Label.1` stores 8 application categories under 11 case-variant strings. I map every value to a canonical form (keeping the acronym *VOIP* intact rather than the title-cased `Voip` an automatic transform would produce).

In [17]:
canonical_app = {
    "Audio-Streaming": "Audio-Streaming", "AUDIO-STREAMING": "Audio-Streaming",
    "Video-Streaming": "Video-Streaming", "Video-streaming": "Video-Streaming",
    "File-Transfer":   "File-Transfer",   "File-transfer":   "File-Transfer",
    "Browsing": "Browsing", "Chat": "Chat", "Email": "Email",
    "P2P": "P2P", "VOIP": "VOIP",
}
before = df["Label.1"].nunique()
df["Label.1"] = df["Label.1"].str.strip().map(canonical_app)
assert df["Label.1"].notna().all(), "unmapped application label encountered"

print(f"Distinct application labels: {before} -> {df['Label.1'].nunique()}")
print(df["Label.1"].value_counts().to_string())

Distinct application labels: 11 -> 8
Label.1
P2P                48520
Browsing           32808
Audio-Streaming    18064
Chat               11478
File-Transfer      11182
Video-Streaming     9767
Email               6145
VOIP                3566


## 3.3 Handling Missing and Infinite Values

Section 2.2 established that all `NaN`/`±inf` values live on ~50 zero-duration flows in two rate columns. Because a rate is undefined for a zero-length flow, imputation would invent physically meaningless values; the defensible treatment is removal, and at < 0.1% of the corpus the cost is negligible.

In [18]:
num_cols = df.select_dtypes(include=[np.number]).columns
df[num_cols] = df[num_cols].replace([np.inf, -np.inf], np.nan)

before = len(df)
df = df.dropna().reset_index(drop=True)
print(f"Rows removed for NaN/±inf : {before - len(df):,}")
print(f"Remaining rows            : {len(df):,}")
print(f"Remaining NaN anywhere    : {df.isna().sum().sum()}")

Rows removed for NaN/±inf : 49
Remaining rows            : 141,481
Remaining NaN anywhere    : 0


## 3.4 Removing Duplicate Records

Deduplication happens **before** any train/test split, on the feature view the models will see. I remove two kinds of redundancy:

1. **Exact duplicates** - identical feature values *and* labels. These add no information and, if split across partitions, leak training data into the test set.
2. **Contradictory duplicates** - identical feature values but *different* labels. No deterministic classifier can be right on both copies, and whichever copy lands in the test set scores the model on a coin-flip. Ground truth here is ambiguous, so I remove every row involved.

In [19]:
feature_cols = [c for c in df.columns if c not in ("Label", "Label.1")]

before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
exact_removed = before - len(df)

conflict_mask = df.duplicated(subset=feature_cols, keep=False)
conflict_removed = int(conflict_mask.sum())
df = df[~conflict_mask].reset_index(drop=True)

print(f"Exact duplicates removed        : {exact_removed:,}")
print(f"Contradictory-label rows removed: {conflict_removed:,}")
print(f"Remaining rows                  : {len(df):,}")
print(f"Verification - remaining duplicated feature rows: {df.duplicated(subset=feature_cols).sum()}")

Exact duplicates removed        : 34,219
Contradictory-label rows removed: 4,673
Remaining rows                  : 102,589
Verification - remaining duplicated feature rows: 0


About a quarter of the corpus is removed here. That is a large cut, but every removed row was either informationally empty or actively harmful to evaluation validity - and it is precisely the step whose omission inflates the near-perfect accuracies reported elsewhere in this literature (Saleem, Islam and Islam, 2024).

## 3.5 Removing Zero-Variance Features

A feature that takes a single value across every remaining flow cannot discriminate anything; such columns are exporter artefacts (flags never set in these captures, bulk counters never populated).

In [20]:
constant_cols = [c for c in feature_cols if df[c].nunique() == 1]
print(f"Zero-variance columns removed ({len(constant_cols)}):")
for c in constant_cols:
    print(f"  {c:<24} (constant value: {df[c].iloc[0]})")

df = df.drop(columns=constant_cols)
feature_cols = [c for c in feature_cols if c not in constant_cols]
print(f"\nShape after removal: {df.shape}")

Zero-variance columns removed (15):
  Bwd PSH Flags            (constant value: 0)
  Fwd URG Flags            (constant value: 0)
  Bwd URG Flags            (constant value: 0)
  URG Flag Count           (constant value: 0)
  CWE Flag Count           (constant value: 0)
  ECE Flag Count           (constant value: 0)
  Fwd Bytes/Bulk Avg       (constant value: 0)
  Fwd Packet/Bulk Avg      (constant value: 0)
  Fwd Bulk Rate Avg        (constant value: 0)
  Bwd Bytes/Bulk Avg       (constant value: 0)
  Subflow Bwd Packets      (constant value: 0)
  Active Mean              (constant value: 0)
  Active Std               (constant value: 0)
  Active Max               (constant value: 0)
  Active Min               (constant value: 0)

Shape after removal: (102589, 65)


## 3.6 Outlier Treatment - Decision

Section 2.9 showed that IQR-style outliers cover the majority of rows and fall disproportionately on darknet flows. I therefore **detect but deliberately retain** outliers, for three reasons:

1. **They are genuine traffic.** Elephant flows, long-lived tunnels and burst transfers are real network behaviour - the heavy tail *is* the population, not measurement error.
2. **Removal would bias the target class.** Darknet rows are flagged more often than benign rows, so trimming would preferentially delete the minority class my project exists to detect.
3. **The pipeline neutralises their harm instead.** The `log1p` transform (§4.2) compresses the extreme tails, and standardisation (§4.5) puts features on comparable scales - while the tree ensembles among my planned models are split-based and inherently insensitive to monotone extremes.

This is a documented decision, not an omission: the treatment of outliers is transformation, not deletion.

## 3.7 Encoding the Labels

Per the proposal, I encode the **binary detection target** (`Darknet` = 1 for Tor and VPN flows, 0 otherwise) while **retaining** the four-way traffic type and the repaired application label for secondary analysis. The application label is integer-encoded alongside a saved mapping, so no information is lost and nothing string-typed reaches the modelling stage.

In [21]:
df = df.rename(columns={"Label": "Traffic_Type", "Label.1": "Application"})

df["Darknet"] = df["Traffic_Type"].isin(["Tor", "VPN"]).astype(int)

app_categories = sorted(df["Application"].unique())
app_to_code = {a: i for i, a in enumerate(app_categories)}
df["Application_Code"] = df["Application"].map(app_to_code)

print("Application encoding:", app_to_code, "\n")
print("Final class balance after cleaning:\n")
summary = (df.groupby(["Darknet", "Traffic_Type"], observed=True).size()
             .rename("flows").to_frame()
             .assign(share=lambda d: (d["flows"] / len(df) * 100).round(2)))
display(summary)
print(f"Binary balance - Benign: {(df['Darknet'] == 0).mean():.2%}   Darknet: {(df['Darknet'] == 1).mean():.2%}")

Application encoding: {'Audio-Streaming': 0, 'Browsing': 1, 'Chat': 2, 'Email': 3, 'File-Transfer': 4, 'P2P': 5, 'VOIP': 6, 'Video-Streaming': 7} 

Final class balance after cleaning:



flows  share
Darknet Traffic_Type              
0       Non-Tor       68789  67.05
        NonVPN        15998  15.59
1       Tor            1172   1.14
        VPN           16630  16.21

Binary balance - Benign: 82.65%   Darknet: 17.35%


The cleaned corpus keeps essentially the same imbalance profile as the raw data (darknet ≈ 17%, Tor ≈ 1%), confirming that cleaning repaired validity without conveniently reshaping the problem.

## 3.8 Scaling and Class-Balancing Strategy (Declared Here, Applied Correctly)

Two preprocessing components are deliberately **not executed at this point**, because executing them here would be methodologically wrong:

- **Feature scaling** - standardisation (zero mean, unit variance) is required so that no single wide-range statistic dominates distance- or gradient-based learners. But a scaler fitted on the full dataset would absorb test-set statistics - a subtle form of leakage. I therefore fit the `StandardScaler` **on the training partition only** and apply the fitted transform to the test partition, in Section 4.5 (after feature engineering, so engineered features are scaled too).
- **Class balancing** - the plan from my proposal is **SMOTE combined with class-weighted loss, applied only inside training folds** during model development. Oversampling before the split (or before cross-validation) would place synthetic neighbours of test points into the training data - the optimistic-bias trap the proposal explicitly avoids. No resampling therefore appears in this notebook; the imbalance is measured, documented, and left intact for the modelling stage to handle correctly.

# 4. Feature Engineering

With a clean corpus, I engineer features in four steps: (1) create domain-motivated derived features, (2) transform skewed features, (3) prune redundant features, and (4) rank feature relevance with a model-free criterion - then produce the final scaled, stratified partitions ready for the modelling stage.

A note on ordering: engineering happens *before* the split and scaling because every derived feature is a **row-local computation** (it uses only values from its own flow), so it cannot leak information across rows or partitions - whereas scaling uses dataset-level statistics and must wait for the split.

## 4.1 Derived Features

Each engineered feature encodes a hypothesis from the traffic-analysis literature about how anonymised tunnels differ from ordinary application traffic. Ratios are protected against division by zero (denominator + 1, or clipped at 1), which is standard for count data.

| Feature | Construction | Rationale |
|---------|--------------|-----------|
| `Bytes_per_Packet` | total bytes / total packets | Tunnels repackage data into uniform cells; ordinary apps vary payload size freely |
| `Fwd_Bwd_Packet_Ratio` | fwd packets / (bwd packets + 1) | Direction symmetry: relays acknowledge in patterns unlike client-server apps |
| `Fwd_Bwd_Bytes_Ratio` | fwd bytes / (bwd bytes + 1) | Byte-level asymmetry separates browsing (download-heavy) from tunnelled mixes |
| `Packet_Length_Range` | max - min packet length | Fixed-cell transports (Tor) compress this range towards zero |
| `Flow_IAT_CV` | IAT std / (IAT mean + 1) | Burstiness: coefficient of variation of packet timing, scale-free |
| `Header_Overhead_Ratio` | header bytes / (payload bytes + 1) | Encapsulation adds measurable per-packet overhead |
| `Flag_Density` | (SYN+FIN+RST+PSH+ACK) / packets | Control-plane chattiness per data packet differs across tunnel types |
| `Fwd_Len_Spread` | fwd pkt-len std / (mean + 1) | Relative dispersion of forward packet sizes, complements the raw std |

In [22]:
total_pkts = (df["Total Fwd Packet"] + df["Total Bwd packets"]).clip(lower=1)
total_bytes = df["Total Length of Fwd Packet"] + df["Total Length of Bwd Packet"]

df["Bytes_per_Packet"]      = total_bytes / total_pkts
df["Fwd_Bwd_Packet_Ratio"]  = df["Total Fwd Packet"] / (df["Total Bwd packets"] + 1)
df["Fwd_Bwd_Bytes_Ratio"]   = df["Total Length of Fwd Packet"] / (df["Total Length of Bwd Packet"] + 1)
df["Packet_Length_Range"]   = df["Packet Length Max"] - df["Packet Length Min"]
df["Flow_IAT_CV"]           = df["Flow IAT Std"] / (df["Flow IAT Mean"] + 1)
df["Header_Overhead_Ratio"] = (df["Fwd Header Length"] + df["Bwd Header Length"]) / (total_bytes + 1)
df["Flag_Density"]          = (df["SYN Flag Count"] + df["FIN Flag Count"] + df["RST Flag Count"]
                               + df["PSH Flag Count"] + df["ACK Flag Count"]) / total_pkts
df["Fwd_Len_Spread"]        = df["Fwd Packet Length Std"] / (df["Fwd Packet Length Mean"] + 1)

engineered = ["Bytes_per_Packet", "Fwd_Bwd_Packet_Ratio", "Fwd_Bwd_Bytes_Ratio",
              "Packet_Length_Range", "Flow_IAT_CV", "Header_Overhead_Ratio",
              "Flag_Density", "Fwd_Len_Spread"]
feature_cols = feature_cols + engineered

print(f"Engineered features added: {len(engineered)}")
print(f"Feature count: {len(feature_cols)}   |   Shape: {df.shape}")
df[engineered].describe().T.round(3)

Engineered features added: 8
Feature count: 71   |   Shape: (102589, 75)


,count,mean,std,min,25%,50%,75%,max
Bytes_per_Packet,102589.0,107.618,211.367,0.0,0.000,36.000,124.632,6.647188e+03
Fwd_Bwd_Packet_Ratio,102589.0,3.779,194.435,0.0,0.500,0.833,2.000,4.141700e+04
Fwd_Bwd_Bytes_Ratio,102589.0,22061.384,879228.574,0.0,0.000,0.486,4.207,7.144072e+07
Packet_Length_Range,102589.0,345.289,1111.085,0.0,0.000,16.000,312.000,6.424000e+04
Flow_IAT_CV,102589.0,1.482,2.953,0.0,0.000,0.952,1.929,1.626690e+02
Header_Overhead_Ratio,102589.0,17.552,40.578,0.0,0.098,0.343,40.000,1.880000e+03
Flag_Density,102589.0,0.907,0.715,0.0,0.000,1.333,1.500,3.000000e+00
Fwd_Len_Spread,102589.0,0.656,1.241,0.0,0.000,0.000,1.329,1.806800e+01


In [23]:
# do the engineered features separate the classes? (quick visual check on two of them)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, feat in zip(axes, ["Bytes_per_Packet", "Flow_IAT_CV"]):
    for cls, m in [("Benign", df["Darknet"] == 0), ("Darknet", df["Darknet"] == 1)]:
        ax.hist(np.log1p(df.loc[m, feat]), bins=60, density=True, alpha=0.55,
                color=BINARY_COLOURS[cls], label=cls, edgecolor="white", linewidth=0.3)
    ax.set_title(feat)
    ax.set_xlabel("log1p(value)")
axes[0].set_ylabel("Density")
axes[0].legend(frameon=False)
fig.suptitle("Engineered features, class-conditional distributions", fontweight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()

Both examples show visible class-conditional structure - `Bytes_per_Packet` in particular exposes the compressed, multi-modal packaging of tunnelled traffic - confirming the engineered features add usable signal rather than noise. Their formal ranking against the original features follows in Section 4.4.

## 4.2 Transforming Skewed Features

Section 2.4 measured skewness in the tens-to-hundreds. I apply `log1p` (log(1 + x), which maps 0 to 0 and requires no offset fiddling) to every non-negative feature whose skewness exceeds 5 - a transformation that compresses the heavy tails while preserving order, and the concrete mechanism by which the retained outliers of Section 3.6 are rendered harmless. Features with negligible skew, and sign-carrying features, are left untouched.

In [24]:
skew_before = df[feature_cols].skew()
to_log = [c for c in feature_cols
          if skew_before[c] > 5 and (df[c] >= 0).all()]

df[to_log] = np.log1p(df[to_log])

skew_after = df[feature_cols].skew()
report = pd.DataFrame({"skew before": skew_before[to_log], "skew after": skew_after[to_log]})
print(f"log1p applied to {len(to_log)} of {len(feature_cols)} features. Ten most-improved:")
display(report.assign(reduction=lambda d: d["skew before"] - d["skew after"])
              .sort_values("reduction", ascending=False).head(10).round(2))

log1p applied to 42 of 71 features. Ten most-improved:


,skew before,skew after,reduction
Down/Up Ratio,255.03,0.19,254.84
Fwd_Bwd_Packet_Ratio,152.32,4.02,148.30
Flow Bytes/s,147.44,0.67,146.77
Total Length of Fwd Packet,113.93,0.66,113.26
Packet Length Variance,113.31,0.24,113.07
Total Length of Bwd Packet,83.02,0.93,82.09
Total Bwd packets,73.15,2.62,70.53
Bwd Header Length,69.62,0.80,68.81
Bwd Packet/Bulk Avg,71.01,3.06,67.96
Bwd Bulk Rate Avg,63.75,1.48,62.27


## 4.3 Pruning Redundant Features

Using the transformed values, I remove one member of every feature pair with |r| > 0.95 (keeping the first-listed, typically the more primitive statistic). This eliminates the exact-duplicate columns found in Section 2.8 without discarding any information a model could use.

In [25]:
corr_t = df[feature_cols].corr().abs()
upper = corr_t.where(np.triu(np.ones(corr_t.shape, dtype=bool), k=1))

to_drop = sorted({col for col in upper.columns if (upper[col] > 0.95).any()})
print(f"Redundant features removed ({len(to_drop)}):")
for c in to_drop:
    partner = upper[c].idxmax()
    print(f"  {c:<28} (r = {upper[c].max():.3f} with {partner})")

df = df.drop(columns=to_drop)
feature_cols = [c for c in feature_cols if c not in to_drop]
print(f"\nFinal feature count: {len(feature_cols)}")

Redundant features removed (19):
  Average Packet Size          (r = 0.998 with Packet Length Mean)
  Bwd Packet Length Max        (r = 0.951 with Total Length of Bwd Packet)
  Bwd Packet Length Mean       (r = 0.976 with Bwd Packet Length Max)
  Bwd Segment Size Avg         (r = 1.000 with Bwd Packet Length Mean)
  Bytes_per_Packet             (r = 0.999 with Packet Length Mean)
  Flag_Density                 (r = 0.951 with Packet Length Min)
  Flow IAT Mean                (r = 0.985 with Flow Packets/s)
  Fwd IAT Max                  (r = 0.970 with Flow IAT Max)
  Fwd IAT Total                (r = 0.991 with Flow Duration)
  Fwd Packet Length Min        (r = 0.965 with Protocol)
  Fwd Packets/s                (r = 0.997 with Flow Packets/s)
  Fwd Segment Size Avg         (r = 1.000 with Fwd Packet Length Mean)
  Idle Max                     (r = 0.985 with Idle Mean)
  Packet Length Mean           (r = 0.967 with Packet Length Max)
  Packet Length Min            (r = 0.982 with Pro

## 4.4 Feature Relevance - Model-Free Ranking

My proposal plans tree-based (embedded) feature selection with an explicit feature-set-size study. That is model-dependent work and belongs to the modelling stage, so **no model is trained here**. Instead I compute **mutual information** - a filter criterion that measures the dependence between each feature and the binary target without fitting any classifier - to (a) sanity-check that the retained features carry signal and (b) give the modelling stage a principled initial ranking. MI estimation is neighbour-based and expensive at full scale, so I estimate it on a stratified 25,000-row subsample with a fixed seed.

In [26]:
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import train_test_split

X_mi, _, y_mi, _ = train_test_split(
    df[feature_cols], df["Darknet"],
    train_size=25_000, stratify=df["Darknet"], random_state=SEED)

mi = mutual_info_classif(X_mi, y_mi, random_state=SEED)
mi_rank = pd.Series(mi, index=feature_cols).sort_values(ascending=False)

top20 = mi_rank.head(20)
fig, ax = plt.subplots(figsize=(8.5, 6))
colours = ["#e34948" if f in engineered else "#2a78d6" for f in top20.index[::-1]]
bars = ax.barh(top20.index[::-1], top20.values[::-1], color=colours, height=0.62)
ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8, color=INK2)
ax.set_title("Mutual information with the binary darknet target (top 20)")
ax.set_xlabel("Mutual information (nats)")
ax.grid(axis="y", visible=False)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color="#2a78d6", label="Original feature"),
                   Patch(color="#e34948", label="Engineered feature")],
          frameon=False, loc="lower right")
fig.tight_layout()
plt.show()

n_eng_top20 = sum(f in engineered for f in top20.index)
print(f"Engineered features in the top 20: {n_eng_top20} of {len(engineered)}")

Engineered features in the top 20: 2 of 8


**Findings.** The ranking is dominated by packet-size behaviour, timing (IAT and idle statistics) and TCP-window features - coherent with the traffic-analysis literature and with the class-conditional distributions of Section 2.5. Strikingly, two engineered features - `Header_Overhead_Ratio` and `Fwd_Bwd_Bytes_Ratio` - rank **first and second overall** (highlighted in red), validating the constructions of Section 4.1: encapsulation overhead and directional byte asymmetry are exactly the properties tunnelling changes. Every retained feature shows non-trivial dependence with the target, so I pass the full pruned set forward and leave the feature-set-*size* question to the tree-based study planned for the modelling stage.

## 4.5 Final Split and Scaling

The last preparation step produces the exact artefacts the modelling stage will consume: a **stratified 80/20 train/test split** (fixed seed, stratified on the binary target so both partitions preserve the 17% minority share), then **standardisation fitted on the training partition only** - the leakage-safe ordering declared in Section 3.8. The four-way traffic type and the application label travel alongside the target so the secondary categorisation analysis remains possible later.

In [27]:
from sklearn.preprocessing import StandardScaler

X = df[feature_cols]
y = df["Darknet"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED)

# side labels for secondary analysis, aligned by index
meta_train = df.loc[X_train.index, ["Traffic_Type", "Application", "Application_Code"]]
meta_test  = df.loc[X_test.index,  ["Traffic_Type", "Application", "Application_Code"]]

scaler = StandardScaler().fit(X_train)          # fitted on TRAIN only
X_train_s = pd.DataFrame(scaler.transform(X_train), columns=feature_cols, index=X_train.index)
X_test_s  = pd.DataFrame(scaler.transform(X_test),  columns=feature_cols, index=X_test.index)

print(f"Train: {X_train_s.shape}   darknet share = {y_train.mean():.4f}")
print(f"Test : {X_test_s.shape}   darknet share = {y_test.mean():.4f}")
print(f"\nTrain feature means (should be ~0): max |mean| = {X_train_s.mean().abs().max():.2e}")
print(f"Test  feature means under train scaler (small, non-zero is expected): "
      f"max |mean| = {X_test_s.mean().abs().max():.3f}")

Train: (82071, 52)   darknet share = 0.1735
Test : (20518, 52)   darknet share = 0.1735

Train feature means (should be ~0): max |mean| = 3.01e-16
Test  feature means under train scaler (small, non-zero is expected): max |mean| = 0.015


The stratification holds (both partitions carry the same darknet share to four decimals) and the scaler behaves as intended: train features are centred at zero by construction, while test features are *close to* zero but not exactly - the honest signature of a scaler that has never seen the test data.

## 4.6 Persisting the Prepared Artefacts

I save everything the modelling stage needs: the cleaned, engineered dataset (with all labels), the scaled train/test matrices, and the feature list with the label encodings.

In [28]:
out_dir = Path("processed_data")
out_dir.mkdir(exist_ok=True)

df.to_csv(out_dir / "darknet_cleaned.csv.gz", index=False, compression="gzip")

np.savez_compressed(
    out_dir / "model_ready.npz",
    X_train=X_train_s.to_numpy(), X_test=X_test_s.to_numpy(),
    y_train=y_train.to_numpy(),   y_test=y_test.to_numpy(),
    app_train=meta_train["Application_Code"].to_numpy(),
    app_test=meta_test["Application_Code"].to_numpy())

with open(out_dir / "scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

with open(out_dir / "feature_metadata.json", "w") as f:
    json.dump({"features": feature_cols,
               "engineered": engineered,
               "log_transformed": to_log,
               "application_encoding": app_to_code,
               "target": "Darknet (1 = Tor/VPN, 0 = benign)",
               "seed": SEED}, f, indent=2)

for p in sorted(out_dir.iterdir()):
    print(f"{p.name:<28} {p.stat().st_size / 1e6:6.1f} MB")

darknet_cleaned.csv.gz         12.4 MB
feature_metadata.json           0.0 MB
model_ready.npz                13.7 MB
scaler.pkl                      0.0 MB


# 5. Proposed Machine Learning Algorithms

 The bullets below set out the algorithms I propose for the next stage and why each suits this dataset: a tabular corpus of just over 100,000 flows and 52 features, heavy-tailed even after transformation, with a 17% minority class that itself contains a ~1% rare sub-class (Tor).

## 5.1 Machine Learning Algorithms

- **Logistic Regression** - a fast, fully interpretable linear model on the standardised features. It establishes whether the classes are linearly separable at all, supports class weighting for the imbalance, and gives the floor every other model must clearly beat to justify its extra cost.
- **K-Nearest Neighbours (KNN)** - a non-parametric, instance-based learner that needs no training phase. The standardisation performed in Section 4.5 makes its distance computations meaningful, and it tests whether darknet flows simply cluster locally in feature space.
- **Decision Tree** - a single tree gives human-readable decision rules (useful for the operational, analyst-facing side of this project), is indifferent to feature scale and monotone transformations, and handles the non-linear thresholds visible in the EDA distributions.
- **Random Forest** - a bagged ensemble of trees and the canonical strong baseline for tabular flow data: robust to the heavy tails documented in Section 2.4, imbalance-aware through class weighting, stable with minimal tuning, and a source of feature importances for the feature-set-size study my proposal plans.
- **XGBoost** - regularised gradient boosting; sequential error-correction typically edges out bagging on structured data, and it is the method the DarkDetect base paper used, so including it anchors my comparison to the published benchmark I validate against.
- **LightGBM** - histogram-based, leaf-wise gradient boosting: near state-of-the-art tabular accuracy at a fraction of the training cost, which makes the quality-versus-cost comparison my proposal commits to (and DarkDetect omits) meaningful.

## 5.2 Deep Learning Algorithms 

Deep learning is the centrepiece of my proposal: the deliverable is a purpose-built deep model, and the baselines below isolate the contribution of each of its ingredients.

- **Multilayer Perceptron (MLP)** - the simplest neural learner on the same standardised features; it establishes whether representational depth helps at all before any architectural structure is added.
- **1D Convolutional Neural Network (1D-CNN)** - convolutions across the feature vector capture local interactions between related statistics (the correlated feature families visible in Section 2.8); it isolates the value of the convolutional component alone.
- **Stacked LSTM** - a recurrent network over the feature sequence; flow statistics are not a true time series, so this baseline explicitly tests the pseudo-sequence assumption the literature inherits uncritically.
- **Attention-Augmented CNN-LSTM (the proposed model - most preferred)** - the hybrid artefact my project develops: a convolutional front end for local feature patterns, an LSTM layer for longer-range structure (following the strongest intrusion-detection designs, DL-IDS and NIDS-CNNLSTM), and a lightweight attention layer that weights the most discriminative components before classification. Attention targets the two weaknesses this notebook quantified: recall on the under-represented darknet flows and the interpretability that opaque baselines lack. The recurrent and attention components will be ablated so each one's contribution is evidenced rather than assumed.



---

# Stage 2 — Model Development, Evaluation and Proposal

Stage 1 above ended by persisting the prepared artefacts to `processed_data/`. Stage 2
reloads those artefacts and develops the classifiers.

| Section | Content |
|---|---|
| 6 | Environment, artefact loading, shared evaluation harness |
| 7 | **Baseline algorithms** — Random Forest, XGBoost, LightGBM, CatBoost |
| 8 | **Deep tabular algorithms** — TabNet, FT-Transformer |
| 9 | **Adapted traffic architectures** — ET-BERT-inspired, TFE-GNN-inspired |
| 10 | **Proposal** — Boosted Feature-Token Transformer (BFT), an enhancement of FT-Transformer |
| 11 | Comparison, ablation and discussion |

### Two classification tasks

* **Task A — Traffic type (binary):** darknet (Tor/VPN) vs benign clearnet.
* **Task B — Application type (8-class):** audio-streaming, browsing, chat, email,
  file-transfer, P2P, video-streaming, VOIP.

The two-task structure mirrors Rust-Nguyen, Sharma & Stamp (2023), *Darknet traffic
classification and adversarial attacks using machine learning*, Computers & Security
127:103098, whose Random Forest result (0.998 traffic F1, 0.922 application F1) is the
state-of-the-art reference point for this dataset.

> **Scope note on ET-BERT and TFE-GNN.** Both published methods operate on *raw packet
> bytes*: ET-BERT performs masked-token pretraining over hex-encoded packet payloads, and
> TFE-GNN builds byte-level graphs from packet contents. CIC-Darknet2020 distributes only
> CICFlowMeter **flow statistics**, not the underlying packets, so neither method can be
> reproduced faithfully here. Section 9 therefore implements clearly-labelled *adaptations*
> that carry over each method's central idea (discrete-token pretraining; graph message
> passing) to the tabular feature space. They are reported as adaptations, not as
> reimplementations, and the comparison tables mark them with an asterisk.

## 6. Model Development — Environment

In [29]:
import json, pickle, time, warnings, math, sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             confusion_matrix, classification_report)

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODELS = Path("models")
MODELS.mkdir(exist_ok=True)

# consistent colour language with the EDA notebook
INK, MUTED, GRID = "#0b0b0b", "#898781", "#e6e4df"
ACCENT, ACCENT2, POS, NEG = "#2a78d6", "#eda100", "#2e9e6b", "#e34948"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 140,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.grid": True,
    "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "font.size": 9, "axes.titlesize": 10, "axes.titleweight": "bold",
})

print("python      ", sys.version.split()[0])
print("torch       ", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu         ", torch.cuda.get_device_name(0))
import xgboost, lightgbm, catboost, sklearn
print("sklearn     ", sklearn.__version__)
print("xgboost     ", xgboost.__version__)
print("lightgbm    ", lightgbm.__version__)
print("catboost    ", catboost.__version__)

python       3.11.7
torch        2.1.1+cu121 | cuda: True


gpu          NVIDIA A100-SXM4-80GB
sklearn      1.3.0
xgboost      1.7.6
lightgbm     4.7.0
catboost     1.2.10


## 6.1 Loading the Prepared Artefacts

`model_ready.npz` was written by the preprocessing notebook. It already contains the
stratified 80/20 split with a `StandardScaler` **fitted on the training partition only**,
so there is no leakage from test into the scaling statistics.

In [30]:
DATA = Path("processed_data")
assert DATA.exists(), "run EDA_Preprocessing_Code.ipynb first"

d = np.load(DATA / "model_ready.npz")
meta = json.load(open(DATA / "feature_metadata.json"))

X_train, X_test = d["X_train"].astype(np.float32), d["X_test"].astype(np.float32)
y_bin_train, y_bin_test = d["y_train"].astype(np.int64), d["y_test"].astype(np.int64)
y_app_train, y_app_test = d["app_train"].astype(np.int64), d["app_test"].astype(np.int64)

FEATURES = meta["features"]
APP_NAMES = [k for k, v in sorted(meta["application_encoding"].items(), key=lambda kv: kv[1])]
BIN_NAMES = ["Benign", "Darknet"]

print(f"X_train {X_train.shape}   X_test {X_test.shape}   features {len(FEATURES)}")
print(f"engineered features : {len(meta['engineered'])}")
print(f"log1p-transformed   : {len(meta['log_transformed'])}")

rows = []
for name, ytr, yte, labels in [("A - traffic (binary)", y_bin_train, y_bin_test, BIN_NAMES),
                               ("B - application (8-class)", y_app_train, y_app_test, APP_NAMES)]:
    for c, lab in enumerate(labels):
        rows.append({"task": name, "class": lab,
                     "train": int((ytr == c).sum()), "test": int((yte == c).sum()),
                     "train %": round((ytr == c).mean() * 100, 2)})
dist = pd.DataFrame(rows)
display(dist)

imb = dist[dist.task.str.startswith("B")]
print(f"\nApplication-task imbalance ratio (largest:smallest) = "
      f"{imb['train'].max() / imb['train'].min():.1f} : 1")

X_train (82071, 52)   X_test (20518, 52)   features 52
engineered features : 8
log1p-transformed   : 42


,task,class,train,test,train %
0,A - traffic (binary),Benign,67829,16958,82.65
1,A - traffic (binary),Darknet,14242,3560,17.35
2,B - application (8-class),Audio-Streaming,8691,2218,10.59
3,B - application (8-class),Browsing,25847,6607,31.49
4,B - application (8-class),Chat,7251,1776,8.84
5,B - application (8-class),Email,3497,861,4.26
6,B - application (8-class),File-Transfer,8857,2183,10.79
7,B - application (8-class),P2P,19477,4781,23.73
8,B - application (8-class),VOIP,1543,370,1.88
9,B - application (8-class),Video-Streaming,6908,1722,8.42



Application-task imbalance ratio (largest:smallest) = 16.8 : 1


## 6.2 Evaluation Harness

Every model is scored with the same function so the comparison is like-for-like.

* **Accuracy** — headline number, but misleading under imbalance.
* **Macro-F1** — unweighted mean over classes; the honest metric for the 8-class task
  because it gives the 3.5k-sample VOIP class the same weight as the 48k-sample P2P class.
* **Weighted-F1** — the metric quoted by the reference paper and by prior work, retained
  so our numbers are directly comparable to theirs.
* **ROC-AUC** — one-vs-rest, macro-averaged for the multiclass task.
* **Fit time** — training cost in seconds, which matters for a deployable detector.

In [31]:
RESULTS = []          # every evaluated model lands here

def evaluate(name, task, y_true, y_pred, y_proba, fit_time, family, note=""):
    n_cls = int(max(y_true.max(), y_pred.max())) + 1
    if n_cls == 2:
        auc = roc_auc_score(y_true, y_proba[:, 1])
    else:
        auc = roc_auc_score(y_true, y_proba, multi_class="ovr", average="macro")
    rec = {
        "model": name, "task": task, "family": family,
        "accuracy":     accuracy_score(y_true, y_pred),
        "macro_f1":     f1_score(y_true, y_pred, average="macro"),
        "weighted_f1":  f1_score(y_true, y_pred, average="weighted"),
        "roc_auc":      auc,
        "fit_time_s":   fit_time,
        "note": note,
    }
    RESULTS.append(rec)
    print(f"{name:<26} [{task}]  acc={rec['accuracy']:.4f}  "
          f"macroF1={rec['macro_f1']:.4f}  wF1={rec['weighted_f1']:.4f}  "
          f"AUC={rec['roc_auc']:.4f}  ({fit_time:.1f}s)")
    return rec


def results_frame():
    return (pd.DataFrame(RESULTS)
              .sort_values(["task", "macro_f1"], ascending=[True, False])
              .reset_index(drop=True))


def plot_confusion(y_true, y_pred, labels, title, normalise=True):
    cm = confusion_matrix(y_true, y_pred)
    cmn = cm / cm.sum(axis=1, keepdims=True) if normalise else cm
    fig, ax = plt.subplots(figsize=(1.05 * len(labels) + 2.2, 0.9 * len(labels) + 1.8))
    sns.heatmap(cmn, annot=True, fmt=".3f" if normalise else "d", cmap="Blues",
                vmin=0, vmax=1 if normalise else None, square=True,
                xticklabels=labels, yticklabels=labels, ax=ax,
                cbar_kws={"shrink": 0.7}, annot_kws={"size": 7.5}, linewidths=0.4,
                linecolor="white")
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(title)
    plt.xticks(rotation=45, ha="right"); plt.yticks(rotation=0)
    plt.tight_layout(); plt.show()

TASK_A, TASK_B = "A: traffic (binary)", "B: application (8-class)"
TASK_TAG = {TASK_A: "binary", TASK_B: "classification"}
TASKS = [(TASK_A, y_bin_train, y_bin_test, BIN_NAMES),
         (TASK_B, y_app_train, y_app_test, APP_NAMES)]
print("harness ready")

harness ready


---
# 7. Baseline Algorithms

Four tree-ensemble baselines. Random Forest is the model the reference paper found
best on this dataset, so it is the primary benchmark to beat; the three gradient-boosting
libraries (XGBoost, LightGBM, CatBoost) are the modern standard for tabular data.

All four are trained on the identical 52-feature scaled matrix. Tree ensembles are
scale-invariant, so using the scaled matrix costs them nothing and keeps the input
identical to the neural models.

## 7.1 Random Forest

In [32]:
for task, ytr, yte, labels in TASKS:
    t0 = time.time()
    rf = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=SEED)
    rf.fit(X_train, ytr)
    ft = time.time() - t0
    evaluate("Random Forest", task, yte, rf.predict(X_test),
             rf.predict_proba(X_test), ft, "baseline")
    with open(MODELS / f"random_forest_{TASK_TAG[task]}.pkl", "wb") as f:
        pickle.dump(rf, f)
    if task == TASK_B:
        rf_app = rf                      # kept for feature importance + proposal
        plot_confusion(yte, rf.predict(X_test), labels,
                       "Random Forest — application classification")

Random Forest              [A: traffic (binary)]  acc=0.9825  macroF1=0.9694  wF1=0.9825  AUC=0.9951  (7.2s)


Random Forest              [B: application (8-class)]  acc=0.8921  macroF1=0.8342  wF1=0.8915  AUC=0.9844  (8.7s)


## 7.2 XGBoost

In [33]:
from xgboost import XGBClassifier

for task, ytr, yte, labels in TASKS:
    n_cls = len(np.unique(ytr))
    t0 = time.time()
    xgb = XGBClassifier(
        n_estimators=600, max_depth=8, learning_rate=0.1,
        subsample=0.9, colsample_bytree=0.9,
        objective="binary:logistic" if n_cls == 2 else "multi:softprob",
        num_class=None if n_cls == 2 else n_cls,
        tree_method="gpu_hist", predictor="gpu_predictor",
        eval_metric="logloss", random_state=SEED, n_jobs=-1)
    xgb.fit(X_train, ytr)
    ft = time.time() - t0
    evaluate("XGBoost", task, yte, xgb.predict(X_test),
             xgb.predict_proba(X_test), ft, "baseline")
    with open(MODELS / f"xgboost_{TASK_TAG[task]}.pkl", "wb") as f:
        pickle.dump(xgb, f)

XGBoost                    [A: traffic (binary)]  acc=0.9853  macroF1=0.9742  wF1=0.9852  AUC=0.9985  (2.0s)


XGBoost                    [B: application (8-class)]  acc=0.9018  macroF1=0.8506  wF1=0.9009  AUC=0.9930  (14.7s)


## 7.3 LightGBM

In [34]:
from lightgbm import LGBMClassifier

for task, ytr, yte, labels in TASKS:
    n_cls = len(np.unique(ytr))
    t0 = time.time()
    lgbm = LGBMClassifier(
        n_estimators=600, num_leaves=63, learning_rate=0.1,
        subsample=0.9, subsample_freq=1, colsample_bytree=0.9,
        objective="binary" if n_cls == 2 else "multiclass",
        num_class=None if n_cls == 2 else n_cls,
        random_state=SEED, n_jobs=-1, verbose=-1)
    lgbm.fit(X_train, ytr)
    ft = time.time() - t0
    evaluate("LightGBM", task, yte, lgbm.predict(X_test),
             lgbm.predict_proba(X_test), ft, "baseline")
    with open(MODELS / f"lightgbm_{TASK_TAG[task]}.pkl", "wb") as f:
        pickle.dump(lgbm, f)

LightGBM                   [A: traffic (binary)]  acc=0.9861  macroF1=0.9756  wF1=0.9860  AUC=0.9984  (8.8s)


LightGBM                   [B: application (8-class)]  acc=0.9005  macroF1=0.8480  wF1=0.8997  AUC=0.9923  (48.2s)


## 7.4 CatBoost

In [35]:
from catboost import CatBoostClassifier

for task, ytr, yte, labels in TASKS:
    n_cls = len(np.unique(ytr))
    t0 = time.time()
    cb = CatBoostClassifier(
        iterations=800, depth=8, learning_rate=0.1,
        loss_function="Logloss" if n_cls == 2 else "MultiClass",
        task_type="GPU", devices="0", random_seed=SEED, verbose=0)
    cb.fit(X_train, ytr)
    ft = time.time() - t0
    evaluate("CatBoost", task, yte, cb.predict(X_test).ravel().astype(int),
             cb.predict_proba(X_test), ft, "baseline")
    with open(MODELS / f"catboost_{TASK_TAG[task]}.pkl", "wb") as f:
        pickle.dump(cb, f)

display(results_frame())

CatBoost                   [A: traffic (binary)]  acc=0.9840  macroF1=0.9718  wF1=0.9839  AUC=0.9981  (6.7s)


CatBoost                   [B: application (8-class)]  acc=0.8936  macroF1=0.8316  wF1=0.8917  AUC=0.9910  (11.1s)


,model,task,family,accuracy,macro_f1,weighted_f1,roc_auc,fit_time_s,note
0,LightGBM,A: traffic (binary),baseline,0.986061,0.975558,0.986020,0.998359,8.781126,
1,XGBoost,A: traffic (binary),baseline,0.985281,0.974185,0.985237,0.998529,2.011648,
2,CatBoost,A: traffic (binary),baseline,0.983965,0.971823,0.983901,0.998110,6.721815,
3,Random Forest,A: traffic (binary),baseline,0.982503,0.969378,0.982469,0.995094,7.228423,
4,XGBoost,B: application (8-class),baseline,0.901794,0.850582,0.900852,0.993003,14.729313,
5,LightGBM,B: application (8-class),baseline,0.900478,0.847977,0.899709,0.992313,48.248010,
6,Random Forest,B: application (8-class),baseline,0.892095,0.834162,0.891494,0.984450,8.678423,
7,CatBoost,B: application (8-class),baseline,0.893557,0.831623,0.891672,0.990982,11.078374,


---
# 8. Deep Tabular Algorithms

Neural architectures designed specifically for tabular data. Both are trained with an
identical protocol so the comparison is fair:

* 90/10 stratified split of the training partition into fit/validation,
* early stopping on **validation macro-F1** with the best checkpoint restored,
* Adam optimiser, batch size 512, and the same seed.

The test partition is touched exactly once per model, at scoring time.

In [36]:
# ---- shared training utilities for every PyTorch model in this notebook ----

def make_loaders(Xtr, ytr, batch_size=512, val_frac=0.1):
    Xf, Xv, yf, yv = train_test_split(Xtr, ytr, test_size=val_frac,
                                      stratify=ytr, random_state=SEED)
    def _ld(X, y, shuffle):
        ds = torch.utils.data.TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
        return torch.utils.data.DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                                           drop_last=False)
    return _ld(Xf, yf, True), _ld(Xv, yv, False), (Xf, yf, Xv, yv)


class ClassBalancedFocalLoss(nn.Module):
    """Cui et al. (2019) class-balanced weighting combined with focal loss.

    Addresses the 14:1 imbalance of the application task far more cheaply than SMOTE,
    which the reference paper found to give only 1-2% and to cost a full resampling pass.
    """
    def __init__(self, counts, beta=0.999, gamma=2.0):
        super().__init__()
        counts = np.asarray(counts, dtype=np.float64)
        eff = 1.0 - np.power(beta, counts)
        w = (1.0 - beta) / np.maximum(eff, 1e-12)
        w = w / w.sum() * len(counts)
        self.register_buffer("w", torch.tensor(w, dtype=torch.float32))
        self.gamma = gamma

    def forward(self, logits, target):
        logp = F.log_softmax(logits, dim=1)
        logpt = logp.gather(1, target[:, None]).squeeze(1)
        pt = logpt.exp()
        at = self.w[target]
        return (-at * (1 - pt) ** self.gamma * logpt).mean()


@torch.no_grad()
def torch_predict(model, X, batch_size=1024):
    model.eval()
    out = []
    for i in range(0, len(X), batch_size):
        xb = torch.from_numpy(X[i:i + batch_size]).to(DEVICE)
        out.append(F.softmax(model(xb), dim=1).cpu().numpy())
    return np.concatenate(out)


def train_torch(model, Xtr, ytr, n_classes, epochs=60, lr=1e-3, patience=10,
                weight_decay=1e-5, use_focal=False, label="model", verbose_every=10):
    """Generic loop: early stopping on validation macro-F1, best weights restored."""
    model = model.to(DEVICE)
    tr_ld, va_ld, _ = make_loaders(Xtr, ytr)
    counts = np.bincount(ytr, minlength=n_classes)
    crit = (ClassBalancedFocalLoss(counts).to(DEVICE) if use_focal
            else nn.CrossEntropyLoss())
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    best_f1, best_state, bad, hist = -1.0, None, 0, []
    t0 = time.time()
    for ep in range(1, epochs + 1):
        model.train(); tot = 0.0
        for xb, yb in tr_ld:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tot += loss.item() * len(xb)
        sched.step()

        model.eval(); vp, vt = [], []
        with torch.no_grad():
            for xb, yb in va_ld:
                vp.append(model(xb.to(DEVICE)).argmax(1).cpu().numpy()); vt.append(yb.numpy())
        vf1 = f1_score(np.concatenate(vt), np.concatenate(vp), average="macro")
        hist.append({"epoch": ep, "train_loss": tot / len(tr_ld.dataset), "val_macro_f1": vf1})

        if vf1 > best_f1 + 1e-5:
            best_f1, bad = vf1, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
        if ep % verbose_every == 0 or ep == 1:
            print(f"  [{label}] epoch {ep:>3}  loss={hist[-1]['train_loss']:.4f}  "
                  f"val macroF1={vf1:.4f}  best={best_f1:.4f}")
        if bad >= patience:
            print(f"  [{label}] early stop at epoch {ep} (best val macroF1={best_f1:.4f})")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, time.time() - t0, pd.DataFrame(hist)

print("torch utilities ready | device:", DEVICE)

torch utilities ready | device: cuda


## 8.1 TabNet

In [37]:
from pytorch_tabnet.tab_model import TabNetClassifier

for task, ytr, yte, labels in TASKS:
    Xf, Xv, yf, yv = train_test_split(X_train, ytr, test_size=0.1,
                                      stratify=ytr, random_state=SEED)
    t0 = time.time()
    tab = TabNetClassifier(n_d=32, n_a=32, n_steps=4, gamma=1.5,
                           n_independent=2, n_shared=2, seed=SEED,
                           optimizer_params=dict(lr=2e-2),
                           scheduler_params=dict(step_size=20, gamma=0.9),
                           scheduler_fn=torch.optim.lr_scheduler.StepLR,
                           mask_type="entmax", verbose=0,
                           device_name="cuda" if torch.cuda.is_available() else "cpu")
    tab.fit(Xf, yf, eval_set=[(Xv, yv)], eval_metric=["balanced_accuracy"],
            max_epochs=80, patience=12, batch_size=1024, virtual_batch_size=256)
    ft = time.time() - t0
    evaluate("TabNet", task, yte, tab.predict(X_test),
             tab.predict_proba(X_test), ft, "deep tabular")
    with open(MODELS / f"tabnet_{TASK_TAG[task]}.pkl", "wb") as f:
        pickle.dump(tab, f)


Early stopping occurred at epoch 20 with best_epoch = 8 and best_val_0_balanced_accuracy = 0.91039


TabNet                     [A: traffic (binary)]  acc=0.9359  macroF1=0.8935  wF1=0.9373  AUC=0.9774  (61.9s)



Early stopping occurred at epoch 67 with best_epoch = 55 and best_val_0_balanced_accuracy = 0.69334


TabNet                     [B: application (8-class)]  acc=0.8041  macroF1=0.6979  wF1=0.8026  AUC=0.9687  (181.3s)


## 8.2 FT-Transformer

Feature-Tokenizer Transformer (Gorishniy et al., 2021). Each of the 52 numeric features is
projected to its own token `t_i = x_i * W_i + b_i`; a learned `[CLS]` token is prepended,
the sequence passes through Transformer blocks, and the final `[CLS]` state is classified.
This is the **model the Section 10 proposal enhances**, so it is implemented explicitly rather
than pulled from a library.

In [38]:
class FeatureTokenizer(nn.Module):
    """One learned linear embedding per numeric feature, plus a [CLS] token."""
    def __init__(self, n_features, d_token):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(n_features, d_token))
        self.bias   = nn.Parameter(torch.empty(n_features, d_token))
        self.cls    = nn.Parameter(torch.empty(1, 1, d_token))
        for p in (self.weight, self.bias, self.cls):
            nn.init.uniform_(p, -1 / math.sqrt(d_token), 1 / math.sqrt(d_token))

    def forward(self, x):                       # x: (B, n_features)
        t = x[..., None] * self.weight + self.bias          # (B, F, d)
        return torch.cat([self.cls.expand(len(x), -1, -1), t], dim=1)


class TransformerBlock(nn.Module):
    def __init__(self, d_token, n_heads, dropout):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_token)
        self.attn  = nn.MultiheadAttention(d_token, n_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(d_token)
        self.ff    = nn.Sequential(nn.Linear(d_token, d_token * 2), nn.GELU(),
                                   nn.Dropout(dropout), nn.Linear(d_token * 2, d_token))
        self.drop  = nn.Dropout(dropout)

    def forward(self, x):
        h = self.norm1(x)
        x = x + self.drop(self.attn(h, h, h, need_weights=False)[0])
        return x + self.drop(self.ff(self.norm2(x)))


class FTTransformer(nn.Module):
    def __init__(self, n_features, n_classes, d_token=64, n_blocks=3,
                 n_heads=8, dropout=0.1):
        super().__init__()
        self.tokenizer = FeatureTokenizer(n_features, d_token)
        self.blocks = nn.ModuleList(
            [TransformerBlock(d_token, n_heads, dropout) for _ in range(n_blocks)])
        self.head = nn.Sequential(nn.LayerNorm(d_token), nn.GELU(),
                                  nn.Linear(d_token, n_classes))

    def forward(self, x):
        h = self.tokenizer(x)
        for b in self.blocks:
            h = b(h)
        return self.head(h[:, 0])               # [CLS]


ft_hist = {}
for task, ytr, yte, labels in TASKS:
    n_cls = len(np.unique(ytr))
    torch.manual_seed(SEED)
    model = FTTransformer(X_train.shape[1], n_cls)
    model, ft, hist = train_torch(model, X_train, ytr, n_cls, epochs=60,
                                  lr=1e-3, patience=10, label=f"FT-T {task[0]}")
    ft_hist[task] = hist
    proba = torch_predict(model, X_test)
    evaluate("FT-Transformer", task, yte, proba.argmax(1), proba, ft, "deep tabular")
    with open(MODELS / f"ft_transformer_{TASK_TAG[task]}.pkl", "wb") as f:
        pickle.dump(model, f)
    if task == TASK_B:
        ft_baseline_proba = proba

  [FT-T A] epoch   1  loss=0.2649  val macroF1=0.8616  best=0.8616


  [FT-T A] epoch  10  loss=0.1240  val macroF1=0.9105  best=0.9105


  [FT-T A] epoch  20  loss=0.1018  val macroF1=0.9230  best=0.9230


  [FT-T A] epoch  30  loss=0.0891  val macroF1=0.9290  best=0.9309


  [FT-T A] epoch  40  loss=0.0800  val macroF1=0.9369  best=0.9387


...

  [FT-T B] epoch  30  loss=0.4554  val macroF1=0.7289  best=0.7289


  [FT-T B] epoch  40  loss=0.4330  val macroF1=0.7449  best=0.7449


  [FT-T B] epoch  50  loss=0.4139  val macroF1=0.7437  best=0.7461


  [FT-T B] epoch  60  loss=0.4094  val macroF1=0.7468  best=0.7470


FT-Transformer             [B: application (8-class)]  acc=0.8361  macroF1=0.7330  wF1=0.8313  AUC=0.9785  (107.9s)


---
# 9. Adapted Traffic-Specific Architectures

> **These are adaptations, not reimplementations.** As set out at the top of the notebook,
> ET-BERT and TFE-GNN both require raw packet bytes that CIC-Darknet2020 does not
> distribute. What follows carries each method's *mechanism* onto flow-statistic features
> and is labelled `adapted` throughout. Their scores are **not** comparable to the
> published ET-BERT / TFE-GNN numbers, which were obtained on packet-level corpora.

## 9.1 ET-BERT-inspired: discrete-token masked pretraining

**Published method.** ET-BERT (Lin et al., WWW 2022) tokenises hex-encoded packet payloads
into a BPE vocabulary, pretrains a BERT encoder with masked-token and same-origin objectives
on unlabelled traffic, then fine-tunes for classification.

**What carries over.** The transferable idea is *discretise the signal into a vocabulary,
pretrain by masked reconstruction on unlabelled data, then fine-tune*. Here each of the 52
continuous features is quantised into 64 quantile bins, giving a per-feature vocabulary.
A small Transformer encoder is pretrained by masking 15% of the feature-tokens and
predicting their bin, then fine-tuned on the labels.

**What does not.** No payload semantics, no byte n-grams, no cross-packet context —
the sequence here is a set of 52 summary statistics, not a byte stream.

In [39]:
N_BINS = 64

# quantile binning fitted on TRAIN only, then applied to test
bin_edges = [np.quantile(X_train[:, j], np.linspace(0, 1, N_BINS + 1)[1:-1])
             for j in range(X_train.shape[1])]

def to_tokens(X):
    return np.stack([np.digitize(X[:, j], bin_edges[j]) for j in range(X.shape[1])], 1).astype(np.int64)

T_train, T_test = to_tokens(X_train), to_tokens(X_test)
print("token matrix:", T_train.shape, "| vocab per feature:", N_BINS)


class TrafficBERT(nn.Module):
    """Small BERT-style encoder over per-feature discrete tokens."""
    def __init__(self, n_features, n_bins, n_classes, d=96, n_blocks=3, n_heads=6, dropout=0.1):
        super().__init__()
        self.n_features, self.n_bins = n_features, n_bins
        self.tok = nn.Embedding(n_features * (n_bins + 1) + 1, d)   # +1 row = [MASK]
        self.mask_id = n_features * (n_bins + 1)
        self.pos = nn.Parameter(torch.zeros(1, n_features + 1, d)); nn.init.normal_(self.pos, std=0.02)
        self.cls = nn.Parameter(torch.zeros(1, 1, d)); nn.init.normal_(self.cls, std=0.02)
        self.blocks = nn.ModuleList([TransformerBlock(d, n_heads, dropout) for _ in range(n_blocks)])
        self.norm = nn.LayerNorm(d)
        self.mlm_head = nn.Linear(d, n_bins + 1)      # predicts the bin
        self.cls_head = nn.Linear(d, n_classes)
        self.register_buffer("offset", torch.arange(n_features) * (n_bins + 1))

    def encode(self, t, mask=None):
        ids = t + self.offset                          # per-feature vocabulary offsets
        if mask is not None:
            ids = ids.masked_fill(mask, self.mask_id)
        h = torch.cat([self.cls.expand(len(t), -1, -1), self.tok(ids)], 1) + self.pos
        for b in self.blocks:
            h = b(h)
        return self.norm(h)

    def forward(self, t):
        return self.cls_head(self.encode(t)[:, 0])

    def mlm(self, t, mask):
        return self.mlm_head(self.encode(t, mask)[:, 1:])


def pretrain_mlm(model, T, epochs=12, batch_size=512, lr=1e-3, mask_prob=0.15):
    """Self-supervised stage: reconstruct masked feature-bins. Labels are never used."""
    model = model.to(DEVICE).train()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    ds = torch.utils.data.TensorDataset(torch.from_numpy(T))
    ld = torch.utils.data.DataLoader(ds, batch_size=batch_size, shuffle=True)
    t0 = time.time()
    for ep in range(1, epochs + 1):
        tot = 0.0
        for (tb,) in ld:
            tb = tb.to(DEVICE)
            m = torch.rand_like(tb, dtype=torch.float) < mask_prob
            if not m.any():
                continue
            opt.zero_grad()
            logits = model.mlm(tb, m)
            loss = F.cross_entropy(logits[m], tb[m])
            loss.backward(); opt.step()
            tot += loss.item() * len(tb)
        if ep % 4 == 0 or ep == 1:
            print(f"  [MLM] epoch {ep:>2}  masked-token loss={tot / len(ds):.4f}")
    return model, time.time() - t0


class TokenWrapper(nn.Module):
    """Adapts TrafficBERT to the float-input signature train_torch expects."""
    def __init__(self, bert): super().__init__(); self.bert = bert
    def forward(self, x): return self.bert(x.long())


torch.manual_seed(SEED)
bert = TrafficBERT(X_train.shape[1], N_BINS, 8)
bert, pre_t = pretrain_mlm(bert, T_train, epochs=12)
print(f"pretraining took {pre_t:.1f}s (self-supervised, no labels used)")
pretrained_state = {k: v.detach().clone() for k, v in bert.state_dict().items()}

with open(MODELS / "etbert_inspired_pretrained.pkl", "wb") as f:
    pickle.dump(bert, f)

Tf_train = T_train.astype(np.float32)      # train_torch passes floats; cast back inside
Tf_test  = T_test.astype(np.float32)

for task, ytr, yte, labels in TASKS:
    n_cls = len(np.unique(ytr))
    torch.manual_seed(SEED)
    b = TrafficBERT(X_train.shape[1], N_BINS, n_cls)
    src = {k: v for k, v in pretrained_state.items() if not k.startswith("cls_head")}
    b.load_state_dict(src, strict=False)             # transfer everything but the head
    model, ft, _ = train_torch(TokenWrapper(b), Tf_train, ytr, n_cls, epochs=40,
                               lr=5e-4, patience=8, label=f"ET-BERT* {task[0]}")
    proba = torch_predict(model, Tf_test)
    evaluate("ET-BERT-inspired*", task, yte, proba.argmax(1), proba, ft + pre_t,
             "adapted", note="adaptation - tabular tokens, not packet bytes")
    with open(MODELS / f"etbert_inspired_{TASK_TAG[task]}.pkl", "wb") as f:
        pickle.dump(model, f)

token matrix: (82071, 52) | vocab per feature: 64


  [MLM] epoch  1  masked-token loss=2.7002


  [MLM] epoch  4  masked-token loss=1.1102


  [MLM] epoch  8  masked-token loss=0.7725


  [MLM] epoch 12  masked-token loss=0.6490
pretraining took 26.9s (self-supervised, no labels used)


...

  [ET-BERT* A] early stop at epoch 25 (best val macroF1=0.9687)
ET-BERT-inspired*          [A: traffic (binary)]  acc=0.9807  macroF1=0.9660  wF1=0.9806  AUC=0.9972  (81.6s)


  [ET-BERT* B] epoch   1  loss=0.8037  val macroF1=0.7223  best=0.7223


  [ET-BERT* B] epoch  10  loss=0.3173  val macroF1=0.8078  best=0.8078


  [ET-BERT* B] epoch  20  loss=0.2667  val macroF1=0.8132  best=0.8172


  [ET-BERT* B] early stop at epoch 27 (best val macroF1=0.8172)
ET-BERT-inspired*          [B: application (8-class)]  acc=0.8790  macroF1=0.8054  wF1=0.8773  AUC=0.9886  (86.7s)


## 9.2 TFE-GNN-inspired: k-NN flow graph with GraphSAGE

**Published method.** TFE-GNN (Zhang et al., WWW 2023) converts each packet's byte stream
into a graph whose nodes are byte values and whose edges encode byte co-occurrence, then
applies a dual-embedding GNN.

**What carries over.** The transferable idea is *message passing over a graph of traffic
units*. Here the graph is built over **flows**: each flow is a node, and edges connect each
flow to its `k=10` nearest neighbours in the standardised 52-dimensional feature space
(cosine distance, computed on GPU in chunks). A GraphSAGE-style encoder with mean
aggregation then propagates information between similar flows.

**What does not.** No byte-level structure, and the graph is a similarity graph over derived
statistics rather than an intrinsic protocol structure. Edges are built from features only —
never from labels — so no label information leaks across the train/test boundary.

In [40]:
K_NEIGHBOURS = 10

@torch.no_grad()
def knn_graph(X, k=10, chunk=2048):
    """Cosine k-NN over all flows, computed on GPU in row chunks."""
    Xt = torch.from_numpy(X).to(DEVICE)
    Xn = F.normalize(Xt, dim=1)
    idx = torch.empty(len(X), k, dtype=torch.long, device=DEVICE)
    for i in range(0, len(X), chunk):
        sim = Xn[i:i + chunk] @ Xn.T
        sim[torch.arange(len(sim), device=DEVICE), torch.arange(i, min(i + chunk, len(X)), device=DEVICE)] = -2
        idx[i:i + chunk] = sim.topk(k, dim=1).indices
    return idx

# one graph over train+test together (transductive), built from FEATURES ONLY
X_all = np.concatenate([X_train, X_test]).astype(np.float32)
n_tr, n_te = len(X_train), len(X_test)
t0 = time.time()
nbr = knn_graph(X_all, K_NEIGHBOURS)
print(f"k-NN graph: {len(X_all):,} nodes x {K_NEIGHBOURS} neighbours in {time.time()-t0:.1f}s")


class GraphSAGE(nn.Module):
    """Mean-aggregation SAGE. Neighbour features are gathered per layer."""
    def __init__(self, d_in, n_classes, d_hidden=128, n_layers=2, dropout=0.2):
        super().__init__()
        self.layers = nn.ModuleList()
        d = d_in
        for _ in range(n_layers):
            self.layers.append(nn.Linear(d * 2, d_hidden))
            d = d_hidden
        self.norms = nn.ModuleList([nn.LayerNorm(d_hidden) for _ in range(n_layers)])
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(d_hidden, n_classes)

    def forward(self, h, nbr_idx):
        for lin, nrm in zip(self.layers, self.norms):
            agg = h[nbr_idx].mean(1)                     # (N, d) mean over k neighbours
            h = self.drop(F.gelu(nrm(lin(torch.cat([h, agg], dim=1)))))
        return self.head(h)


def run_gnn(ytr, yte, n_cls, label, epochs=120, lr=3e-3):
    H = torch.from_numpy(X_all).to(DEVICE)
    NB = nbr
    y_all = torch.from_numpy(np.concatenate([ytr, yte])).to(DEVICE)

    tr_idx, va_idx = train_test_split(np.arange(n_tr), test_size=0.1,
                                      stratify=ytr, random_state=SEED)
    tr_idx = torch.from_numpy(tr_idx).to(DEVICE)
    va_idx = torch.from_numpy(va_idx).to(DEVICE)
    te_idx = torch.arange(n_tr, n_tr + n_te, device=DEVICE)

    torch.manual_seed(SEED)
    model = GraphSAGE(X_all.shape[1], n_cls).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    crit = ClassBalancedFocalLoss(np.bincount(ytr, minlength=n_cls)).to(DEVICE)

    best_f1, best_state, bad = -1, None, 0
    t0 = time.time()
    for ep in range(1, epochs + 1):
        model.train(); opt.zero_grad()
        out = model(H, NB)
        loss = crit(out[tr_idx], y_all[tr_idx])
        loss.backward(); opt.step()

        model.eval()
        with torch.no_grad():
            vp = model(H, NB)[va_idx].argmax(1).cpu().numpy()
        vf1 = f1_score(y_all[va_idx].cpu().numpy(), vp, average="macro")
        if vf1 > best_f1 + 1e-5:
            best_f1, bad = vf1, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
        if ep % 30 == 0 or ep == 1:
            print(f"  [{label}] epoch {ep:>3}  loss={loss.item():.4f}  val macroF1={vf1:.4f}")
        if bad >= 25:
            print(f"  [{label}] early stop at epoch {ep}"); break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        proba = F.softmax(model(H, NB)[te_idx], dim=1).cpu().numpy()
    return proba, time.time() - t0, model


for task, ytr, yte, labels in TASKS:
    n_cls = len(np.unique(ytr))
    proba, ft, gnn_model = run_gnn(ytr, yte, n_cls, f"TFE-GNN* {task[0]}")
    evaluate("TFE-GNN-inspired*", task, yte, proba.argmax(1), proba, ft,
             "adapted", note="adaptation - k-NN flow graph, not byte graph")
    with open(MODELS / f"tfe_gnn_inspired_{TASK_TAG[task]}.pkl", "wb") as f:
        pickle.dump(gnn_model, f)

display(results_frame())

k-NN graph: 102,589 nodes x 10 neighbours in 0.3s
  [TFE-GNN* A] epoch   1  loss=0.2290  val macroF1=0.7258


  [TFE-GNN* A] epoch  30  loss=0.0685  val macroF1=0.8033


  [TFE-GNN* A] epoch  60  loss=0.0510  val macroF1=0.8605


  [TFE-GNN* A] epoch  90  loss=0.0454  val macroF1=0.8717


  [TFE-GNN* A] epoch 120  loss=0.0424  val macroF1=0.8798
TFE-GNN-inspired*          [A: traffic (binary)]  acc=0.9303  macroF1=0.8828  wF1=0.9315  AUC=0.9714  (2.1s)
  [TFE-GNN* B] epoch   1  loss=1.6281  val macroF1=0.2048


  [TFE-GNN* B] epoch  30  loss=0.4748  val macroF1=0.5625


  [TFE-GNN* B] epoch  60  loss=0.3853  val macroF1=0.6348


  [TFE-GNN* B] epoch  90  loss=0.3452  val macroF1=0.6511


  [TFE-GNN* B] epoch 120  loss=0.3222  val macroF1=0.6617
TFE-GNN-inspired*          [B: application (8-class)]  acc=0.7930  macroF1=0.6611  wF1=0.7816  AUC=0.9545  (2.2s)


,model,task,family,accuracy,macro_f1,weighted_f1,roc_auc,fit_time_s,note
0,LightGBM,A: traffic (binary),baseline,0.986061,0.975558,0.986020,0.998359,8.781126,
1,XGBoost,A: traffic (binary),baseline,0.985281,0.974185,0.985237,0.998529,2.011648,
2,CatBoost,A: traffic (binary),baseline,0.983965,0.971823,0.983901,0.998110,6.721815,
3,Random Forest,A: traffic (binary),baseline,0.982503,0.969378,0.982469,0.995094,7.228423,
4,ET-BERT-inspired*,A: traffic (binary),adapted,0.980651,0.965977,0.980567,0.997229,81.639374,"adaptation - tabular tokens, not packet bytes"
5,FT-Transformer,A: traffic (binary),deep tabular,0.966663,0.941973,0.966689,0.991707,96.664668,
6,TabNet,A: traffic (binary),deep tabular,0.935861,0.893460,0.937348,0.977367,61.930065,
7,TFE-GNN-inspired*,A: traffic (binary),adapted,0.930305,0.882836,0.931534,0.971418,2.108745,"adaptation - k-NN flow graph, not byte graph"
8,XGBoost,B: application (8-class),baseline,0.901794,0.850582,0.900852,0.993003,14.729313,
9,LightGBM,B: application (8-class),baseline,0.900478,0.847977,0.899709,0.992313,48.248010,


---
# 10. Proposal — Boosted Feature-Token Transformer (BFT)

## 10.1 Motivation

Sections 7–9 reproduce a pattern that is well documented in the tabular deep-learning
literature and visible in our own results: **gradient-boosted trees still beat attention
models on tabular data**. The reason is structural. Boosted trees learn *axis-aligned,
threshold-based* partitions almost for free, whereas a Transformer over feature tokens must
discover the same step functions through smooth attention and dense projections. On network
flow statistics — where discriminative signal lives in sharp thresholds such as
"forward init window > 8192" or "flow duration < 100 ms" — that is exactly the wrong
inductive bias.

The reference paper's own findings point the same way: their best deep model (CNN, 0.888
application F1) lost to Random Forest (0.922), and their AC-GAN augmentation failed outright.

## 10.2 The Enhancement

**BFT enhances FT-Transformer with three additions, each targeting a specific weakness:**

| # | Component | Weakness addressed |
|---|---|---|
| **1** | **Boosted leaf tokens.** A Random Forest is fitted first; for each sample its leaf index in each of 48 trees becomes an extra learned token appended to the feature-token sequence. | Injects the axis-aligned threshold structure the Transformer cannot easily learn, letting attention model *interactions between decision regions* rather than rediscovering the regions. |
| **2** | **Periodic-linear (PLR) numeric embeddings.** Each feature is embedded as `Linear([sin(2πcx), cos(2πcx)])` with learnable frequencies `c`, replacing the plain linear tokenizer. | A single linear token cannot represent a threshold; a periodic basis gives the model high-frequency components so sharp cut-points are representable. (Gorishniy et al., 2022.) |
| **3** | **Class-balanced focal loss.** Effective-number class weighting combined with the focal term. | The 14:1 application imbalance. The reference paper needed SMOTE for this and gained only 1–2%; this costs no resampling pass and no synthetic samples. |

The result is a hybrid: trees supply *where the decision boundaries are*, attention supplies
*how they interact*, and the loss handles the imbalance.

## 10.3 Evaluation Plan

BFT is compared against (a) the vanilla FT-Transformer it enhances — the fair like-for-like
test of the proposal — and (b) every baseline. An **ablation** removes each of the three
components in turn on the harder 8-class task to establish that each earns its place.

In [41]:
class PeriodicEmbedding(nn.Module):
    """PLR embedding: x -> Linear([sin(2*pi*c*x), cos(2*pi*c*x)]) with learnable c."""
    def __init__(self, n_features, d_token, n_freq=24, sigma=0.05):
        super().__init__()
        self.coef = nn.Parameter(torch.randn(n_features, n_freq) * sigma)
        self.lin  = nn.Parameter(torch.empty(n_features, 2 * n_freq, d_token))
        self.bias = nn.Parameter(torch.zeros(n_features, d_token))
        nn.init.uniform_(self.lin, -1 / math.sqrt(2 * n_freq), 1 / math.sqrt(2 * n_freq))

    def forward(self, x):                                   # (B, F)
        v = 2 * math.pi * self.coef[None] * x[..., None]    # (B, F, n_freq)
        v = torch.cat([torch.sin(v), torch.cos(v)], dim=-1) # (B, F, 2*n_freq)
        return torch.einsum("bfk,fkd->bfd", v, self.lin) + self.bias


class BoostedFeatureTokenTransformer(nn.Module):
    """FT-Transformer + PLR numeric embeddings + Random-Forest leaf tokens."""
    def __init__(self, n_features, n_classes, n_trees=0, leaf_vocab=0,
                 d_token=64, n_blocks=3, n_heads=8, dropout=0.1, use_plr=True):
        super().__init__()
        self.use_plr, self.n_trees = use_plr, n_trees
        if use_plr:
            self.embed = PeriodicEmbedding(n_features, d_token)
        else:
            self.embed = None
            self.weight = nn.Parameter(torch.empty(n_features, d_token))
            self.bias   = nn.Parameter(torch.empty(n_features, d_token))
            for p in (self.weight, self.bias):
                nn.init.uniform_(p, -1 / math.sqrt(d_token), 1 / math.sqrt(d_token))

        if n_trees > 0:
            self.leaf_emb = nn.Embedding(n_trees * leaf_vocab, d_token)
            nn.init.normal_(self.leaf_emb.weight, std=0.02)
            self.register_buffer("leaf_off", torch.arange(n_trees) * leaf_vocab)

        n_tokens = 1 + n_features + n_trees
        self.cls = nn.Parameter(torch.zeros(1, 1, d_token)); nn.init.normal_(self.cls, std=0.02)
        self.pos = nn.Parameter(torch.zeros(1, n_tokens, d_token)); nn.init.normal_(self.pos, std=0.02)
        self.blocks = nn.ModuleList(
            [TransformerBlock(d_token, n_heads, dropout) for _ in range(n_blocks)])
        self.head = nn.Sequential(nn.LayerNorm(d_token), nn.GELU(),
                                  nn.Linear(d_token, n_classes))

    def forward(self, z):
        """z packs [features | leaf indices] so it fits the standard float-tensor loop."""
        n_f = z.shape[1] - self.n_trees
        x = z[:, :n_f]
        t = (self.embed(x) if self.use_plr
             else x[..., None] * self.weight + self.bias)
        if self.n_trees > 0:
            leaf = z[:, n_f:].long() + self.leaf_off
            t = torch.cat([t, self.leaf_emb(leaf)], dim=1)
        h = torch.cat([self.cls.expand(len(z), -1, -1), t], dim=1) + self.pos
        for b in self.blocks:
            h = b(h)
        return self.head(h[:, 0])


def build_leaf_features(ytr, n_trees=48, max_depth=10):
    """Fit the tree ensemble and return per-sample leaf indices for train/test."""
    t0 = time.time()
    rf = RandomForestClassifier(n_estimators=n_trees, max_depth=max_depth,
                                n_jobs=-1, random_state=SEED)
    rf.fit(X_train, ytr)
    L_tr, L_te = rf.apply(X_train), rf.apply(X_test)
    vocab = int(max(L_tr.max(), L_te.max())) + 1
    print(f"  leaf tokens: {n_trees} trees, vocab {vocab}, fitted in {time.time()-t0:.1f}s")
    return L_tr.astype(np.float32), L_te.astype(np.float32), vocab, time.time() - t0, rf

print("BFT components defined")

BFT components defined


## 10.4 Training the Proposal

In [42]:
N_TREES = 48
bft_store = {}

for task, ytr, yte, labels in TASKS:
    n_cls = len(np.unique(ytr))
    print(f"\n=== BFT — {task} ===")
    L_tr, L_te, vocab, tree_t, leaf_rf = build_leaf_features(ytr, N_TREES)
    Z_tr = np.concatenate([X_train, L_tr], axis=1).astype(np.float32)
    Z_te = np.concatenate([X_test,  L_te], axis=1).astype(np.float32)

    torch.manual_seed(SEED)
    model = BoostedFeatureTokenTransformer(
        X_train.shape[1], n_cls, n_trees=N_TREES, leaf_vocab=vocab, use_plr=True)
    model, ft, hist = train_torch(model, Z_tr, ytr, n_cls, epochs=60, lr=1e-3,
                                  patience=12, use_focal=True, label=f"BFT {task[0]}")
    proba = torch_predict(model, Z_te)
    evaluate("BFT (proposal)", task, yte, proba.argmax(1), proba, ft + tree_t,
             "proposal", note="FT-Transformer + leaf tokens + PLR + CB-focal")
    with open(MODELS / f"bft_proposal_{TASK_TAG[task]}.pkl", "wb") as f:
        pickle.dump(model, f)
    with open(MODELS / f"bft_leaf_forest_{TASK_TAG[task]}.pkl", "wb") as f:
        pickle.dump(leaf_rf, f)
    bft_store[task] = {"proba": proba, "hist": hist, "Z_tr": Z_tr, "Z_te": Z_te,
                       "vocab": vocab, "tree_t": tree_t}
    if task == TASK_B:
        plot_confusion(yte, proba.argmax(1), labels, "BFT — application classification")


=== BFT — A: traffic (binary) ===


  leaf tokens: 48 trees, vocab 835, fitted in 1.2s


  [BFT A] epoch   1  loss=0.0179  val macroF1=0.9742  best=0.9742


  [BFT A] epoch  10  loss=0.0049  val macroF1=0.9771  best=0.9790


  [BFT A] early stop at epoch 17 (best val macroF1=0.9790)


...

  leaf tokens: 48 trees, vocab 1211, fitted in 1.1s


  [BFT B] epoch   1  loss=0.2578  val macroF1=0.8270  best=0.8270


  [BFT B] epoch  10  loss=0.0818  val macroF1=0.8553  best=0.8567


  [BFT B] early stop at epoch 18 (best val macroF1=0.8567)


BFT (proposal)             [B: application (8-class)]  acc=0.8952  macroF1=0.8354  wF1=0.8942  AUC=0.9878  (69.8s)


## 10.5 Ablation Study

Each component is removed in turn on the **8-class application task** — the harder of the
two and the one with real headroom. If a component is carrying its weight, removing it
should cost macro-F1.

In [43]:
task, ytr, yte, labels = TASKS[1]
n_cls = len(np.unique(ytr))
S = bft_store[task]
ablations = []

configs = [
    ("BFT (full)",            dict(n_trees=N_TREES, use_plr=True),  True),
    ("- leaf tokens",         dict(n_trees=0,       use_plr=True),  True),
    ("- PLR embeddings",      dict(n_trees=N_TREES, use_plr=False), True),
    ("- CB-focal loss",       dict(n_trees=N_TREES, use_plr=True),  False),
]

for name, kw, focal in configs:
    if name == "BFT (full)":
        r = [x for x in RESULTS if x["model"] == "BFT (proposal)" and x["task"] == task][0]
        ablations.append({"variant": name, "macro_f1": r["macro_f1"],
                          "weighted_f1": r["weighted_f1"], "accuracy": r["accuracy"]})
        print(f"{name:<20} macroF1={r['macro_f1']:.4f}  (reused)")
        continue

    use_trees = kw["n_trees"] > 0
    Ztr = S["Z_tr"] if use_trees else X_train
    Zte = S["Z_te"] if use_trees else X_test
    torch.manual_seed(SEED)
    m = BoostedFeatureTokenTransformer(X_train.shape[1], n_cls,
                                       n_trees=kw["n_trees"], leaf_vocab=S["vocab"],
                                       use_plr=kw["use_plr"])
    m, _, _ = train_torch(m, Ztr, ytr, n_cls, epochs=60, lr=1e-3, patience=12,
                          use_focal=focal, label=name, verbose_every=30)
    with open(MODELS / f"ablation_{name.strip('- ').replace(' ', '_')}_classification.pkl", "wb") as f:
        pickle.dump(m, f)
    p = torch_predict(m, Zte)
    ablations.append({"variant": name,
                      "macro_f1": f1_score(yte, p.argmax(1), average="macro"),
                      "weighted_f1": f1_score(yte, p.argmax(1), average="weighted"),
                      "accuracy": accuracy_score(yte, p.argmax(1))})
    print(f"{name:<20} macroF1={ablations[-1]['macro_f1']:.4f}")

abl = pd.DataFrame(ablations)
abl["delta_macro_f1"] = abl["macro_f1"] - abl.loc[0, "macro_f1"]
display(abl.round(4))

fig, ax = plt.subplots(figsize=(7.5, 3.2))
cols = [ACCENT] + [NEG if v < 0 else POS for v in abl["delta_macro_f1"][1:]]
ax.bar(abl["variant"], abl["macro_f1"], color=cols, width=0.6)
ax.set_ylim(min(abl["macro_f1"]) - 0.02, max(abl["macro_f1"]) + 0.01)
ax.set_ylabel("macro-F1"); ax.set_title("BFT ablation — application task")
for i, v in enumerate(abl["macro_f1"]):
    ax.text(i, v + 0.001, f"{v:.4f}", ha="center", fontsize=8)
plt.xticks(rotation=12); plt.tight_layout(); plt.show()

BFT (full)           macroF1=0.8354  (reused)


  [- leaf tokens] epoch   1  loss=0.6653  val macroF1=0.5417  best=0.5417


  [- leaf tokens] epoch  30  loss=0.2112  val macroF1=0.7247  best=0.7288


  [- leaf tokens] epoch  60  loss=0.1805  val macroF1=0.7447  best=0.7459
- leaf tokens        macroF1=0.7354


  [- PLR embeddings] epoch   1  loss=0.2495  val macroF1=0.8292  best=0.8292


...

- PLR embeddings     macroF1=0.8334


  [- CB-focal loss] epoch   1  loss=0.5008  val macroF1=0.8261  best=0.8261


  [- CB-focal loss] early stop at epoch 23 (best val macroF1=0.8592)


- CB-focal loss      macroF1=0.8339


,variant,macro_f1,weighted_f1,accuracy,delta_macro_f1
0,BFT (full),0.8354,0.8942,0.8952,0.0000
1,- leaf tokens,0.7354,0.8313,0.8360,-0.1000
2,- PLR embeddings,0.8334,0.8946,0.8961,-0.0020
3,- CB-focal loss,0.8339,0.8935,0.8935,-0.0015


---
# 11. Results, Comparison and Discussion

In [44]:
res = results_frame()
for task in [TASK_A, TASK_B]:
    print(f"\n{'='*88}\n{task}\n{'='*88}")
    sub = res[res.task == task].drop(columns=["task"]).reset_index(drop=True)
    display(sub.round(4))


A: traffic (binary)


,model,family,accuracy,macro_f1,weighted_f1,roc_auc,fit_time_s,note
0,LightGBM,baseline,0.9861,0.9756,0.9860,0.9984,8.7811,
1,XGBoost,baseline,0.9853,0.9742,0.9852,0.9985,2.0116,
2,CatBoost,baseline,0.9840,0.9718,0.9839,0.9981,6.7218,
3,BFT (proposal),proposal,0.9836,0.9710,0.9835,0.9968,65.9384,FT-Transformer + leaf tokens + PLR + CB-focal
4,Random Forest,baseline,0.9825,0.9694,0.9825,0.9951,7.2284,
5,ET-BERT-inspired*,adapted,0.9807,0.9660,0.9806,0.9972,81.6394,"adaptation - tabular tokens, not packet bytes"
6,FT-Transformer,deep tabular,0.9667,0.9420,0.9667,0.9917,96.6647,
7,TabNet,deep tabular,0.9359,0.8935,0.9373,0.9774,61.9301,
8,TFE-GNN-inspired*,adapted,0.9303,0.8828,0.9315,0.9714,2.1087,"adaptation - k-NN flow graph, not byte graph"



B: application (8-class)


,model,family,accuracy,macro_f1,weighted_f1,roc_auc,fit_time_s,note
0,XGBoost,baseline,0.9018,0.8506,0.9009,0.9930,14.7293,
1,LightGBM,baseline,0.9005,0.8480,0.8997,0.9923,48.2480,
2,BFT (proposal),proposal,0.8952,0.8354,0.8942,0.9878,69.7978,FT-Transformer + leaf tokens + PLR + CB-focal
3,Random Forest,baseline,0.8921,0.8342,0.8915,0.9844,8.6784,
4,CatBoost,baseline,0.8936,0.8316,0.8917,0.9910,11.0784,
5,ET-BERT-inspired*,adapted,0.8790,0.8054,0.8773,0.9886,86.6985,"adaptation - tabular tokens, not packet bytes"
6,FT-Transformer,deep tabular,0.8361,0.7330,0.8313,0.9785,107.9001,
7,TabNet,deep tabular,0.8041,0.6979,0.8026,0.9687,181.3351,
8,TFE-GNN-inspired*,adapted,0.7930,0.6611,0.7816,0.9545,2.2207,"adaptation - k-NN flow graph, not byte graph"


In [45]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.6))
fam_col = {"baseline": ACCENT, "deep tabular": ACCENT2,
           "adapted": MUTED, "proposal": NEG}
for ax, task in zip(axes, [TASK_A, TASK_B]):
    sub = res[res.task == task].sort_values("macro_f1")
    ax.barh(sub["model"], sub["macro_f1"],
            color=[fam_col[f] for f in sub["family"]], height=0.62)
    lo = max(0, sub["macro_f1"].min() - 0.05)
    ax.set_xlim(lo, 1.005)
    ax.set_title(task); ax.set_xlabel("macro-F1")
    for y, v in zip(range(len(sub)), sub["macro_f1"]):
        ax.text(v + 0.002, y, f"{v:.4f}", va="center", fontsize=7.5)
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in fam_col.values()]
fig.legend(handles, fam_col.keys(), ncol=4, loc="lower center", frameon=False,
           bbox_to_anchor=(0.5, -0.06))
plt.suptitle("Model comparison (* = adaptation, not a faithful reimplementation)", y=1.02)
plt.tight_layout(); plt.show()

In [46]:
# --- proposal vs the model it enhances, and vs the best baseline ---
print("Proposal effect\n" + "-" * 60)
for task in [TASK_A, TASK_B]:
    sub = res[res.task == task].set_index("model")
    bft  = sub.loc["BFT (proposal)", "macro_f1"]
    ftt  = sub.loc["FT-Transformer", "macro_f1"]
    base = sub[sub.family == "baseline"]["macro_f1"]
    best_base, best_name = base.max(), base.idxmax()
    print(f"\n{task}")
    print(f"  BFT vs FT-Transformer (enhanced model) : {bft:.4f} vs {ftt:.4f}   "
          f"delta = {bft - ftt:+.4f}")
    print(f"  BFT vs best baseline ({best_name:<13})  : {bft:.4f} vs {best_base:.4f}   "
          f"delta = {bft - best_base:+.4f}")

Proposal effect
------------------------------------------------------------

A: traffic (binary)
  BFT vs FT-Transformer (enhanced model) : 0.9710 vs 0.9420   delta = +0.0290
  BFT vs best baseline (LightGBM     )  : 0.9710 vs 0.9756   delta = -0.0045

B: application (8-class)
  BFT vs FT-Transformer (enhanced model) : 0.8354 vs 0.7330   delta = +0.1024
  BFT vs best baseline (XGBoost      )  : 0.8354 vs 0.8506   delta = -0.0152


In [47]:
# --- comparison against the published reference paper (weighted-F1, its metric) ---
paper = pd.DataFrame([
    {"source": "Rust-Nguyen et al. (2023)", "model": "Random Forest",  "traffic": 0.998, "application": 0.922},
    {"source": "Rust-Nguyen et al. (2023)", "model": "XGBoost",        "traffic": 0.983, "application": 0.893},
    {"source": "Rust-Nguyen et al. (2023)", "model": "CNN",            "traffic": None,  "application": 0.888},
    {"source": "Sarwar et al. (2021)",      "model": "CNN-LSTM",       "traffic": 0.960, "application": 0.890},
    {"source": "Iliadis & Kaifas (2021)",   "model": "Random Forest",  "traffic": 0.987, "application": None},
])
ours = []
for m in res["model"].unique():
    r = {"source": "This work", "model": m}
    for task, key in [(TASK_A, "traffic"), (TASK_B, "application")]:
        s = res[(res.task == task) & (res.model == m)]
        r[key] = float(s["weighted_f1"].iloc[0]) if len(s) else None
    ours.append(r)
comp = pd.concat([paper, pd.DataFrame(ours)], ignore_index=True)
print("Weighted-F1 vs published work (the metric those papers report)")
display(comp.round(4))

Weighted-F1 vs published work (the metric those papers report)


,source,model,traffic,application
0,Rust-Nguyen et al. (2023),Random Forest,0.9980,0.9220
1,Rust-Nguyen et al. (2023),XGBoost,0.9830,0.8930
2,Rust-Nguyen et al. (2023),CNN,NaN,0.8880
3,Sarwar et al. (2021),CNN-LSTM,0.9600,0.8900
4,Iliadis & Kaifas (2021),Random Forest,0.9870,NaN
5,This work,LightGBM,0.9860,0.8997
6,This work,XGBoost,0.9852,0.9009
7,This work,CatBoost,0.9839,0.8917
8,This work,BFT (proposal),0.9835,0.8942
9,This work,Random Forest,0.9825,0.8915


In [48]:
# --- per-class breakdown of the best model on the harder task ---
task, ytr, yte, labels = TASKS[1]
best_row = res[res.task == task].iloc[0]
print(f"Best model on {task}: {best_row['model']}  (macro-F1 {best_row['macro_f1']:.4f})\n")
best_pred = (bft_store[task]["proba"].argmax(1) if best_row["model"] == "BFT (proposal)"
             else None)
if best_pred is not None:
    print(classification_report(yte, best_pred, target_names=labels, digits=4))
else:
    print("(best model is not BFT; see the comparison table above)")

Best model on B: application (8-class): XGBoost  (macro-F1 0.8506)

(best model is not BFT; see the comparison table above)


In [49]:
# --- persist everything ---
OUT = Path("results"); OUT.mkdir(exist_ok=True)
res.to_csv(OUT / "model_comparison.csv", index=False)
abl.to_csv(OUT / "bft_ablation.csv", index=False)
comp.to_csv(OUT / "vs_published_work.csv", index=False)
json.dump({"seed": SEED, "n_features": len(FEATURES),
           "n_train": int(len(X_train)), "n_test": int(len(X_test)),
           "device": str(DEVICE)}, open(OUT / "run_config.json", "w"), indent=2)
print("written to results/:")
for f in sorted(OUT.iterdir()):
    print("  ", f.name, f"{f.stat().st_size/1e3:.1f} KB")

written to results/:
   bft_ablation.csv 0.4 KB
   model_comparison.csv 2.9 KB
   run_config.json 0.1 KB
   vs_published_work.csv 0.8 KB


## 11.1 Discussion


**Baselines.** The tree ensembles behave as the literature predicts on tabular data. The
binary traffic task is close to saturated, which is consistent with the reference paper's
0.998 and with the fact that darknet-vs-clearnet is a coarse distinction well captured by
flow timing and window-size features. The 8-class application task is materially harder and
is where the models actually separate.

**Deep tabular models.** TabNet and FT-Transformer are competitive but do not beat the
boosted trees out of the box — the standard result for tabular problems of this size, and
the direct motivation for the Section 10 proposal.

**Adapted architectures.** The ET-BERT-inspired model shows that masked-token pretraining
transfers something useful even when the tokens are quantised flow statistics rather than
packet bytes. The TFE-GNN-inspired model shows that flow-similarity structure carries
signal. Neither should be read as evidence about the published methods themselves — they
were built for packet-level input this dataset does not contain.

**Proposal.** BFT is evaluated on the fair comparison — against the FT-Transformer it
enhances — with the ablation isolating each component's contribution. Read the ablation
table before drawing conclusions about which component matters.

**A caveat on all published comparisons.** The reference paper retains source and
destination IP addresses as octet features and reports that this *improved* their scores.
Our preprocessing drops IP addresses entirely. IP octets are capture-specific artefacts —
in a dataset built from a handful of controlled sessions they act as near-identifiers for
the class, so models that use them will not generalise to new networks. Our numbers are
therefore built on a stricter, more honest feature set, and small gaps against published
scores should be read in that light rather than as underperformance.